<a href="https://colab.research.google.com/github/AntonDozhdikov/politpredict/blob/main/World_Politics_MARL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 World Political System as a Multi-Agent Reinforcement Learning System
## Modelling on World Bank WDI & WGI Data

**Abstract.**  
This notebook implements a full computational pipeline for studying the world
political system as a multi-agent system with reinforcement learning.  
Countries act as agents operating in a shared environment parameterised by
World Development Indicators (WDI) and Worldwide Governance Indicators (WGI).

**Pipeline stages:**
1. Data loading & preprocessing (WDI CSV + WGI Excel)
2. Exploratory analysis, country clustering, vulnerability index
3. Baseline forecasting (ETS / Holt-Winters per indicator per country)
4. Ensemble GRU World Model (NLL training, uncertainty estimation)
5. Multi-Agent RL – MADDPG (cooperative baseline)
6. Multi-Agent RL – MATD3 (twin-critic, policy delay)
7. Evolutionary Self-Referential fine-tuning – Evo-MATD3-SR
8. Algorithm comparison & hypothesis testing (12 political hypotheses)
9. Country-level scenario analysis incl. Russia (RUS), G7, BRICS
10. Crisis stress-tests (financial, energy, sanctions, pandemic)
11. Trajectory simulation 2025-2050
12. Final dashboard & export

**Data sources:**
- `wdi_panel_raw.csv` — World Bank WDI panel (1990-2023, ~180 countries, 38 indicators)
- `wgidataset_with_sourcedata-2025.xlsx` — WGI 2025 (sheets: va, pv, ge, rq, rl, cc)

> **Runtime:** Google Colab (GPU recommended).  
> Run: *Runtime → Run all*  
> All output tables are saved to `output/tables/*.csv` (GitHub-exportable).  
> All figures are saved to `output/figures/*.png`.  
> Checkpoints in `output/checkpoints/*.pt`.


## Cell 1 · Dependencies & Imports

In [ ]:
# ── 1. Dependencies ─────────────────────────────────────────────────────────
import subprocess, sys

PKGS = ["statsmodels", "tqdm", "openpyxl", "ipywidgets"]
for p in PKGS:
    try:
        __import__(p)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])

print("✓ packages OK")

# ── 2. Core imports ──────────────────────────────────────────────────────────
import os, copy, time, json, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error

from statsmodels.tsa.holtwinters import ExponentialSmoothing

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm, trange

# ipywidgets for Colab-compatible tables
import ipywidgets as widgets
from IPython.display import display, HTML

# ── 3. Reproducibility ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


✓ packages OK
DEVICE: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 2 · Configuration

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
from dataclasses import dataclass, field


@dataclass
class Config:
    # ── Data paths ──────────────────────────────────────────────────────────
    WDI_PATH:      str = "wdi_panel_raw.csv"
    WGI_PATH:      str = "wgidataset_with_sourcedata-2025.xlsx"
    OUTPUT_DIR:    str = "output"
    SEED:          int = 42


    # ── Train/val/test split ─────────────────────────────────────────────────
    YEAR_TRAIN_END: int = 2010   # train: up to (incl.)
    YEAR_VAL_END:   int = 2016   # val:   2011-2016
    #                               test:  2017+
    FORECAST_HORIZON: int = 26   # 2025-2050


    # ── State space (41 features = 35 WDI + 6 WGI) ──────────────────────────
    WDI_FEATURES: List[str] = field(default_factory=lambda: [
        # macroeconomics
        "gdp_growth", "gdp_pc_usd", "gdp_pc_growth",
        "inflation", "unemployment",
        "gross_capital_form", "govt_consumption",
        "exports_gdp", "imports_gdp", "trade_gdp",
        "fdi_gdp", "manufacturing_gdp", "nat_resources_gdp",
        # innovation & knowledge
        "rd_gdp", "researchers_pm", "journal_articles",
        "patents_residents", "hitech_exports",
        # digitalisation
        "internet_users", "mobile_subs", "broadband", "secure_servers_pm",
        # social & demographic
        "life_expectancy", "infant_mortality",
        "school_secondary", "school_tertiary", "edu_gdp",
        "labor_participation", "female_labor", "gini_index",
        "pop_growth",
        # urban & ecology
        "urban_pop_pct", "renewable_energy",
        "electricity_access", "energy_use_pc",
    ])
    WGI_FEATURES: List[str] = field(default_factory=lambda: [
        "va_score", "pv_score", "ge_score",
        "rq_score", "rl_score", "cc_score",
    ])


    # ACTION_DIM = 14 political-economic instruments
    ACTION_NAMES: List[str] = field(default_factory=lambda: [
        "trade_openness",        # A01 – reduce tariffs, trade agreements
        "military_buildup",      # A02 – increase defence spending
        "rnd_investment",        # A03 – boost R&D and innovation
        "digitalisation",        # A04 – ICT infrastructure
        "education_policy",      # A05 – secondary/tertiary enrolment
        "social_equity",         # A06 – reduce gini, female labour
        "climate_energy",        # A07 – renewables, energy efficiency
        "infrastructure",        # A08 – electricity access, capital formation
        "fiscal_stimulus",       # A09 – govt consumption, capital form
        "fdi_attraction",        # A10 – investment climate
        "institution_reform",    # A11 – improve GE/RQ/RL/CC
        "stability_security",    # A12 – improve PV (political stability)
        "alliance_cooperation",  # A13 – deepen regional/global cooperation
        "sanctions_pressure",    # A14 – impose costs on selected partners
    ])


    # ── Derived dimensions (set after data load) ─────────────────────────────
    STATE_DIM:  int = 41    # 35 WDI + 6 WGI (+ optional income/region OHE)
    ACTION_DIM: int = 14
    BUDGET_LIMIT: float = 4.0   # sum of action vector ≤ BUDGET_LIMIT


    # ── Reward weights ───────────────────────────────────────────────────────
    REWARD_WEIGHTS: Dict[str, float] = field(default_factory=lambda: {
        "gdp_growth":        0.18,
        "governance_mean":   0.18,   # mean of 6 WGI scores (normalised 0-1)
        "life_expectancy":   0.10,
        "trade_gdp":         0.10,
        "unemployment":      0.08,   # penalise high unemployment
        "infant_mortality":  0.07,   # penalise high IM
        "gini_index":        0.07,   # penalise high inequality
        "renewable_energy":  0.07,
        "internet_users":    0.05,
        "military_budget":   0.05,   # mild penalty for very high A02
        "budget_penalty":    0.05,
    })


    # ── Crisis scenarios ─────────────────────────────────────────────────────
    # (4-vector: gdp_shock, mortality_shock, migration_shock, energy_shock)
    CRISIS_INJECT_PROB: float = 0.15
    TAIL_RISK_PROB:     float = 0.04


    # ── World Model (Ensemble GRU) ───────────────────────────────────────────
    WM_ENSEMBLE_SIZE: int   = 3
    WM_HIDDEN_DIM:    int   = 128
    WM_N_LAYERS:      int   = 2
    WM_DROPOUT:       float = 0.15
    WM_LR:            float = 3e-4
    WM_EPOCHS:        int   = 1000
    WM_BATCH_SIZE:    int   = 256
    WM_PATIENCE:      int   = 100


    # ── MADDPG ───────────────────────────────────────────────────────────────
    # усиленный, но совместимый набор: только значения
    MADDPG_EPISODES:     int   = 1800    # 1000 → 1800 (длиннее обучение)
    MADDPG_EPISODE_LEN:  int   = 18      # 15 → 18 (чуть длиннее горизонт)
    MADDPG_BATCH_SIZE:   int   = 256
    MADDPG_BUFFER_SIZE:  int   = 100000  # 50000 → 100000 (больше разнообразия)
    MADDPG_LR_ACTOR:     float = 3e-4    # 1e-4 → 3e-4 (стандартный шаг для DDPG)
    MADDPG_LR_CRITIC:    float = 3e-4    # 3e-4 (оставили, но симметризовали)
    MADDPG_GAMMA:        float = 0.985   # 0.97 → 0.985 (более длинный горизонт)
    MADDPG_TAU:          float = 0.005
    MADDPG_WARMUP:       int   = 800     # 300 → 800 (дольше собираем буфер)
    MADDPG_HIDDEN:       int   = 256
    MADDPG_LOG_INTERVAL: int   = 50


    # ── MATD3 ────────────────────────────────────────────────────────────────
    # параллельные изменения, также только значения
    MATD3_EPISODES:      int   = 1800    # 1000 → 1800
    MATD3_LR_ACTOR:      float = 3e-4    # 1e-4 → 3e-4
    MATD3_LR_CRITIC:     float = 3e-4    # 3e-4 (симметрия с MADDPG)
    MATD3_GAMMA:         float = 0.985   # 0.97 → 0.985
    MATD3_TAU:           float = 0.005
    MATD3_POLICY_DELAY:  int   = 1       # 2 → 1 (чуть менее консервативный TD3)
    MATD3_TARGET_NOISE:  float = 0.15    # 0.10 → 0.15
    MATD3_NOISE_CLIP:    float = 0.25    # 0.20 → 0.25
    MATD3_HIDDEN:        int   = 256
    MATD3_LOG_INTERVAL:  int   = 50


    # ── Evo-MATD3-SR ─────────────────────────────────────────────────────────
    # больше итераций и популяция, но структура та же
    EVO_NITERS:         int   = 150   # 60 → 150 (глубже поиск)
    EVO_POP_SIZE:       int   = 32    # 16 → 32 (богаче популяция)
    EVO_ELITE_FRAC:     float = 0.30
    EVO_BIAS_STD_INIT:  float = 0.06  # 0.04 → 0.06 (чуть шире стартовый шаг)
    EVO_BIAS_STD_MIN:   float = 0.015 # 0.005 → 0.015 (не даём σ упасть до нуля)
    EVO_BIAS_STD_DECAY: float = 0.985 # 0.97 → 0.985 (медленнее затухание)
    EVO_EVAL_FAST:      int   = 4     # 3 → 4 (чуть точнее быстрая оценка)
    EVO_EVAL_REFINE:    int   = 8     # 6 → 8 (чуть глубже refine)


    # ── Early stopping (shared) ──────────────────────────────────────────────
    RL_PATIENCE: int = 250   # 100 → 250 (позже ранняя остановка)


    # ── Figures ──────────────────────────────────────────────────────────────
    FIGURE_STYLE: str = "seaborn-v0_8-whitegrid"
    FIGURE_DPI:   int = 120


CFG = Config()


# Create output dirs
for d in ["output/checkpoints","output/figures","output/tables","output/logs"]:
    Path(d).mkdir(parents=True, exist_ok=True)


print(f"  State dim (initial):  {CFG.STATE_DIM}")
print(f"  Action dim:           {CFG.ACTION_DIM}")
print(f"  Budget limit:         {CFG.BUDGET_LIMIT}")
print(f"  Action names: {CFG.ACTION_NAMES}")
print(f"  MADDPG/MATD3/Evo episodes: "
      f"{CFG.MADDPG_EPISODES}/{CFG.MATD3_EPISODES}/{CFG.EVO_NITERS}")

  State dim (initial):  41
  Action dim:           14
  Budget limit:         4.0
  Action names: ['trade_openness', 'military_buildup', 'rnd_investment', 'digitalisation', 'education_policy', 'social_equity', 'climate_energy', 'infrastructure', 'fiscal_stimulus', 'fdi_attraction', 'institution_reform', 'stability_security', 'alliance_cooperation', 'sanctions_pressure']
  MADDPG/MATD3/Evo episodes: 1800/1800/150


## Cell 3 · Utility Classes (Timer, EarlyStopping, Logger, ArtifactManager)

In [ ]:
# ── Timer ────────────────────────────────────────────────────────────────────
class Timer:
    def __init__(self, name: str):
        self.name = name
    def __enter__(self):
        self.t0 = time.time(); return self
    def __exit__(self, *_):
        print(f"  ⏱  {self.name}: {time.time()-self.t0:.1f}s")

# ── EarlyStopping ─────────────────────────────────────────────────────────────
class EarlyStopping:
    """Monitors a score; stops when no improvement for `patience` steps."""
    def __init__(self, patience=20, mode="min", delta=1e-5, restore_best=True):
        self.patience = patience
        self.mode = mode
        self.delta = delta
        self.restore_best = restore_best
        self.best_score = np.inf if mode == "min" else -np.inf
        self.counter = 0
        self.best_state = None

    def _is_better(self, score):
        if self.mode == "min":
            return score < self.best_score - self.delta
        return score > self.best_score + self.delta

    def step(self, score, model: Optional[nn.Module] = None) -> bool:
        """Returns True if training should stop."""
        if self._is_better(score):
            self.best_score = score
            self.counter = 0
            if model is not None and self.restore_best:
                self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        if self.counter >= self.patience:
            if model is not None and self.best_state is not None and self.restore_best:
                model.load_state_dict(self.best_state)
            return True
        return False

# ── ExperimentLogger ──────────────────────────────────────────────────────────
class ExperimentLogger:
    """Stores episode metrics as JSON-serialisable records."""
    def __init__(self, name: str):
        self.name = name
        self.records: List[Dict] = []

    @staticmethod
    def _ser(obj):
        if isinstance(obj, dict):   return {str(k): ExperimentLogger._ser(v) for k,v in obj.items()}
        if isinstance(obj, (list, tuple)): return [ExperimentLogger._ser(v) for v in obj]
        if isinstance(obj, np.ndarray):    return obj.tolist()
        if isinstance(obj, np.floating):   return float(obj)
        if isinstance(obj, np.integer):    return int(obj)
        if isinstance(obj, np.bool_):      return bool(obj)
        return obj

    def log(self, episode: int, metrics: Dict[str, float]):
        self.records.append({"episode": int(episode), **self._ser(metrics)})

    def save(self, path=None):
        path = path or f"output/logs/{self.name}_log.json"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self._ser(self.records), f, indent=2, ensure_ascii=False)

    def to_df(self):
        return pd.DataFrame(self.records)

# ── ArtifactManager ───────────────────────────────────────────────────────────
class ArtifactManager:
    @staticmethod
    def save_table(df: pd.DataFrame, name: str) -> str:
        path = f"output/tables/{name}.csv"
        df.to_csv(path, index=True)
        return path

    @staticmethod
    def save_figure(fig, name: str, dpi: int = 120) -> str:
        path = f"output/figures/{name}.png"
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return path

    @staticmethod
    def show_table(df: pd.DataFrame, title: str = "", max_rows: int = 50):
        """Render DataFrame as an HTML widget (visible in Colab & exported HTML)."""
        sub = df.head(max_rows)
        html = sub.to_html(border=1, classes="dataframe",
                           float_format=lambda x: f"{x:.4f}")
        header = f"<h4>{title}</h4>" if title else ""
        display(HTML(header + html))

AM = ArtifactManager()
print("✓ Utility classes ready")


✓ Utility classes ready


## Cell 4 · Data Loading: WDI + WGI → panel dataframe

In [ ]:
# ── 4.1 Load WDI ─────────────────────────────────────────────────────────────
with Timer("Load WDI"):
    wdi_raw = pd.read_csv(CFG.WDI_PATH, low_memory=False)
    # Normalise column names
    wdi_raw.columns = [c.strip().lower().replace(" ","_") for c in wdi_raw.columns]
    for c in wdi_raw.columns:
        if c not in ("country_code","country_name","year"):
            wdi_raw[c] = pd.to_numeric(wdi_raw[c], errors="coerce")
    wdi_raw = wdi_raw.sort_values(["country_code","year"]).reset_index(drop=True)
    print(f"  WDI shape: {wdi_raw.shape}  |  countries: {wdi_raw['country_code'].nunique()}"
          f"  |  years: {wdi_raw['year'].min()}-{wdi_raw['year'].max()}")

# ── 4.2 Load WGI Excel (sheets: va, pv, ge, rq, rl, cc) ──────────────────────
WGI_DIMS = ["va","pv","ge","rq","rl","cc"]

with Timer("Load WGI"):
    wgi_parts = []
    for dim in WGI_DIMS:
        try:
            sheet = pd.read_excel(CFG.WGI_PATH, sheet_name=dim, header=0)
            # Normalise column names
            sheet.columns = [str(c).strip().lower()
                             .replace(" ","_").replace("(","").replace(")","")
                             .replace("/","_") for c in sheet.columns]
            # Keep economy code, year and governance score (0-100)
            # Possible score column names:
            score_col = None
            for cname in sheet.columns:
                if "governance_score" in cname and "0-100" in cname:
                    score_col = cname; break
                if cname in ("governance_score_0-100","governance_score","score_0_100"):
                    score_col = cname; break
            if score_col is None:
                # Try positional: col index 12 per WGI documentation
                score_col = sheet.columns[12] if len(sheet.columns) > 12 else None

            econ_col = None
            for cname in sheet.columns:
                if "economy_code" in cname or cname == "economy_code":
                    econ_col = cname; break
            if econ_col is None:
                for cname in sheet.columns:
                    if "code" in cname:
                        econ_col = cname; break

            year_col = "year" if "year" in sheet.columns else None

            if econ_col and year_col and score_col:
                sub = sheet[[econ_col, year_col, score_col]].copy()
                sub.columns = ["country_code","year", f"{dim}_score"]
                sub["country_code"] = sub["country_code"].astype(str).str.strip()
                sub["year"] = pd.to_numeric(sub["year"], errors="coerce")
                sub[f"{dim}_score"] = pd.to_numeric(sub[f"{dim}_score"], errors="coerce")
                sub = sub.dropna(subset=["country_code","year"])
                wgi_parts.append(sub)
                print(f"  Sheet '{dim}': {len(sub)} rows, score col='{score_col}'")
            else:
                print(f"  ⚠️  Sheet '{dim}': could not identify columns "
                      f"(econ={econ_col}, year={year_col}, score={score_col})")
                print(f"     Available columns: {list(sheet.columns[:10])}")
        except Exception as ex:
            print(f"  ⚠️  Sheet '{dim}' error: {ex}")

    if wgi_parts:
        wgi_panel = wgi_parts[0]
        for part in wgi_parts[1:]:
            wgi_panel = wgi_panel.merge(part, on=["country_code","year"], how="outer")
        print(f"  WGI panel shape: {wgi_panel.shape}")
    else:
        print("  ⚠️  WGI could not be loaded – creating empty placeholder")
        wgi_panel = pd.DataFrame(columns=["country_code","year"] +
                                 [f"{d}_score" for d in WGI_DIMS])

# ── 4.3 Merge WDI + WGI ──────────────────────────────────────────────────────
with Timer("Merge WDI+WGI"):
    panel = wdi_raw.merge(wgi_panel, on=["country_code","year"], how="left")
    print(f"  Merged panel: {panel.shape}")
    # Available feature columns
    ALL_FEAT_COLS = [f for f in CFG.WDI_FEATURES + CFG.WGI_FEATURES
                     if f in panel.columns]
    print(f"  Available state features: {len(ALL_FEAT_COLS)} / "
          f"{len(CFG.WDI_FEATURES)+len(CFG.WGI_FEATURES)}")
    missing = [f for f in CFG.WDI_FEATURES + CFG.WGI_FEATURES
               if f not in panel.columns]
    if missing:
        print(f"  Missing (will skip): {missing}")

# ── 4.4 Coverage audit ───────────────────────────────────────────────────────
coverage = (panel.groupby("country_code")[ALL_FEAT_COLS]
            .apply(lambda g: g.notna().mean().mean())
            .sort_values(ascending=False))
GOOD_COUNTRIES = coverage[coverage >= 0.30].index.tolist()
panel = panel[panel["country_code"].isin(GOOD_COUNTRIES)].copy()
print(f"  Countries with ≥30% feature coverage: {len(GOOD_COUNTRIES)}")

audit_df = coverage.reset_index().rename(columns={0:"mean_coverage"})
AM.save_table(audit_df, "00_country_coverage")
AM.show_table(audit_df.head(30), "Country coverage (top-30)")
print(f"  Final panel: {panel.shape}")


  WDI shape: (6076, 39)  |  countries: 217  |  years: 1996-2023
  ⏱  Load WDI: 0.1s
  Sheet 'va': 5444 rows, score col='governance_score_0-100'
  Sheet 'pv': 5429 rows, score col='governance_score_0-100'
  Sheet 'ge': 5340 rows, score col='governance_score_0-100'
  Sheet 'rq': 5341 rows, score col='governance_score_0-100'
  Sheet 'rl': 5476 rows, score col='governance_score_0-100'
  Sheet 'cc': 5373 rows, score col='governance_score_0-100'
  WGI panel shape: (32403, 8)
  ⏱  Load WGI: 16.2s
  Merged panel: (6076, 45)
  Available state features: 41 / 41
  ⏱  Merge WDI+WGI: 0.0s
  Countries with ≥30% feature coverage: 208


,country_code,mean_coverage
0,ESP,0.8101
1,POL,0.8101
2,LTU,0.8092
3,LVA,0.8092
4,ROU,0.8084
5,CZE,0.8066
6,EST,0.8049
7,HUN,0.8049
8,GBR,0.8040
9,DNK,0.8040


  Final panel: (5824, 45)


## Cell 5 · Preprocessing: imputation, log-transforms, scaling, adjacency

In [ ]:
# ── 5.1 Log-transforms for skewed indicators ─────────────────────────────────
with Timer("Log-transforms"):
    for col in [
        "gdp_pc_usd", "researchers_pm", "journal_articles", "patents_residents",
        "mobile_subs", "secure_servers_pm", "infant_mortality"
    ]:
        if col in panel.columns:
            panel[col] = pd.to_numeric(panel[col], errors="coerce")
            panel[col] = np.log1p(panel[col].clip(lower=0))

    # ── 5.2 Composite: mean governance score ─────────────────────────────────
    wgi_present = [
        c for c in ["va_score", "pv_score", "ge_score", "rq_score", "rl_score", "cc_score"]
        if c in panel.columns
    ]
    if wgi_present:
        for c in wgi_present:
            panel[c] = pd.to_numeric(panel[c], errors="coerce")
        panel["governance_mean"] = panel[wgi_present].mean(axis=1)
    else:
        panel["governance_mean"] = np.nan

# ── 5.3 Sanitize feature list BEFORE interpolation/imputation ────────────────
# 1) keep only existing columns
ALL_FEAT_COLS = [f for f in ALL_FEAT_COLS if f in panel.columns]

# 2) remove duplicates while preserving order
ALL_FEAT_COLS = list(dict.fromkeys(ALL_FEAT_COLS))

# 3) force numeric
for col in ALL_FEAT_COLS:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

print(f"  Candidate features before filtering: {len(ALL_FEAT_COLS)}")

# ── 5.4 Time interpolation per country ───────────────────────────────────────
def fill_ts(grp):
    grp = grp.sort_values("year").copy()
    grp[ALL_FEAT_COLS] = (
        grp[ALL_FEAT_COLS]
        .interpolate(method="linear", limit_direction="both")
        .ffill()
        .bfill()
    )
    return grp

with Timer("Interpolation"):
    panel = panel.groupby("country_code", group_keys=False).apply(fill_ts)

# ── 5.5 Remove fully empty columns BEFORE imputation ─────────────────────────
non_empty_feat_cols = [c for c in ALL_FEAT_COLS if panel[c].notna().any()]
dropped_empty_cols = [c for c in ALL_FEAT_COLS if c not in non_empty_feat_cols]

if dropped_empty_cols:
    print(f"  Dropping fully empty features ({len(dropped_empty_cols)}): {dropped_empty_cols}")

ALL_FEAT_COLS = non_empty_feat_cols

# ── 5.6 Impute remaining NaN safely ──────────────────────────────────────────
with Timer("Imputation"):
    imp = SimpleImputer(strategy="median")
    imputed_array = imp.fit_transform(panel[ALL_FEAT_COLS])

    imputed_df = pd.DataFrame(
        imputed_array,
        columns=ALL_FEAT_COLS,
        index=panel.index
    )

    panel.loc[:, ALL_FEAT_COLS] = imputed_df

# ── 5.7 Income-group one-hot (optional structural feature) ──────────────────
INCOME_COLS = []
if "income_group" in panel.columns:
    ohe_inc = pd.get_dummies(panel["income_group"], prefix="inc").astype(float)
    INCOME_COLS = sorted(ohe_inc.columns.tolist())
    panel = pd.concat(
        [panel.reset_index(drop=True), ohe_inc.reset_index(drop=True)],
        axis=1
    )
    print(f"  Income OHE: {INCOME_COLS}")
else:
    print("  income_group not found -> skipping income one-hot")

# ── 5.8 Final feature list & RobustScaler ────────────────────────────────────
FEAT_COLS = ALL_FEAT_COLS + INCOME_COLS
FEAT_COLS = list(dict.fromkeys(FEAT_COLS))

CFG.STATE_DIM = len(FEAT_COLS)

scaler = RobustScaler()
df_norm = panel.copy()
df_norm.loc[:, FEAT_COLS] = scaler.fit_transform(df_norm[FEAT_COLS])

print(f"  Final STATE_DIM = {CFG.STATE_DIM}")

# ── 5.9 Train/Val/Test split ─────────────────────────────────────────────────
df_train = df_norm[df_norm["year"] <= CFG.YEAR_TRAIN_END].copy()
df_val   = df_norm[
    (df_norm["year"] > CFG.YEAR_TRAIN_END) &
    (df_norm["year"] <= CFG.YEAR_VAL_END)
].copy()
df_test  = df_norm[df_norm["year"] > CFG.YEAR_VAL_END].copy()

print(f"  Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}")

# ── 5.10 Country list & index ────────────────────────────────────────────────
countries = sorted(df_norm["country_code"].unique())
N_COUNTRIES = len(countries)
CTRY_IDX = {c: i for i, c in enumerate(countries)}
print(f"  Countries in model: {N_COUNTRIES}")

# ── 5.11 Adjacency matrix (WGI cluster proximity fallback) ───────────────────
adj_matrix = np.zeros((N_COUNTRIES, N_COUNTRIES), dtype=np.float32)

if wgi_present:
    wgi_profile = (
        df_norm.groupby("country_code")[wgi_present]
        .mean()
        .reindex(countries)
        .fillna(0)
    )
    km_adj = KMeans(n_clusters=8, n_init=20, random_state=SEED)
    clust_labels_adj = km_adj.fit_predict(wgi_profile)

    for i, c1 in enumerate(countries):
        for j, c2 in enumerate(countries):
            if i != j and clust_labels_adj[i] == clust_labels_adj[j]:
                adj_matrix[i, j] = 1.0
else:
    adj_matrix = np.ones((N_COUNTRIES, N_COUNTRIES), dtype=np.float32)
    np.fill_diagonal(adj_matrix, 0.0)
    clust_labels_adj = np.full(N_COUNTRIES, -1)

row_sum = adj_matrix.sum(axis=1, keepdims=True).clip(min=1)
ADJ_NORM = adj_matrix / row_sum
ADJ_TENSOR = torch.tensor(ADJ_NORM, dtype=torch.float32, device=DEVICE)

print(f"  Adjacency {N_COUNTRIES}×{N_COUNTRIES}, non-zero edges: {int(adj_matrix.sum())}")

# save cluster table
AM.save_table(
    pd.DataFrame({
        "country": countries,
        "cluster": clust_labels_adj
    }),
    "01_country_clusters_adj"
)

print("✓ Preprocessing done")

  ⏱  Log-transforms: 0.0s
  Candidate features before filtering: 41
  ⏱  Interpolation: 1.9s
  Dropping fully empty features (6): ['va_score', 'pv_score', 'ge_score', 'rq_score', 'rl_score', 'cc_score']
  ⏱  Imputation: 0.0s
  income_group not found -> skipping income one-hot
  Final STATE_DIM = 35
  Train: 3120  Val: 1248  Test: 1456
  Countries in model: 208
  Adjacency 208×208, non-zero edges: 43056
✓ Preprocessing done


## Cell 6 · Exploratory Analysis: country clustering & vulnerability index

In [ ]:
# ── 6.1 Country profiles & PCA ────────────────────────────────────────────────
with Timer("EDA / Clustering"):
    # Берем только реально существующие числовые признаки
    feat_for_profile = [c for c in FEAT_COLS if c in df_norm.columns]
    feat_for_profile = list(dict.fromkeys(feat_for_profile))

    reg_profile = (
        df_norm.groupby("country_code")[feat_for_profile]
        .mean()
        .replace([np.inf, -np.inf], np.nan)
    )

    # Удаляем полностью пустые колонки
    reg_profile = reg_profile.dropna(axis=1, how="all")

    # На всякий случай медианная импутация по страновым профилям
    if reg_profile.isna().sum().sum() > 0:
        reg_profile = reg_profile.apply(lambda s: s.fillna(s.median()), axis=0)

    # Признаки для PCA/кластеризации
    cluster_feats = reg_profile.columns.tolist()
    X_country = reg_profile[cluster_feats].values

    # PCA
    pca = PCA(n_components=2, random_state=SEED)
    X_pca = pca.fit_transform(X_country)

    # KMeans clustering
    km = KMeans(n_clusters=7, n_init=30, random_state=SEED)
    cluster_labels = km.fit_predict(X_country)
    reg_profile["cluster"] = cluster_labels

    print(f"  {len(np.unique(cluster_labels))} clusters, PCA var explained: "
          f"{pca.explained_variance_ratio_.sum()*100:.1f}%")

    # ── Vulnerability index ───────────────────────────────────────────────────
    vuln_components = []

    if "inflation_cpi" in reg_profile.columns:
        vuln_components.append(reg_profile["inflation_cpi"].rank(pct=True))
    if "unemployment" in reg_profile.columns:
        vuln_components.append(reg_profile["unemployment"].rank(pct=True))
    if "gini_index" in reg_profile.columns:
        vuln_components.append(reg_profile["gini_index"].rank(pct=True))
    if "infant_mortality" in reg_profile.columns:
        vuln_components.append(reg_profile["infant_mortality"].rank(pct=True))
    if "life_expectancy" in reg_profile.columns:
        vuln_components.append(1 - reg_profile["life_expectancy"].rank(pct=True))
    if "governance_mean" in reg_profile.columns:
        vuln_components.append(1 - reg_profile["governance_mean"].rank(pct=True))
    elif len([c for c in wgi_present if c in reg_profile.columns]) > 0:
        gov_tmp = reg_profile[[c for c in wgi_present if c in reg_profile.columns]].mean(axis=1)
        vuln_components.append(1 - gov_tmp.rank(pct=True))

    if len(vuln_components) > 0:
        reg_profile["vulnerability_idx"] = pd.concat(vuln_components, axis=1).mean(axis=1)
    else:
        reg_profile["vulnerability_idx"] = 0.5

    # Maps for later cells
    CLUSTER_MAP = reg_profile["cluster"].to_dict()
    VULN_MAP = reg_profile["vulnerability_idx"].to_dict()

# ── 6.2 Plot: country clusters in PCA space ───────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
pal7 = sns.color_palette("tab10", n_colors=7)

for cl in sorted(reg_profile["cluster"].unique()):
    mask = reg_profile["cluster"] == cl
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=70,
        alpha=0.8,
        color=pal7[int(cl)],
        label=f"Cluster {cl}",
        edgecolor="black",
        linewidth=0.4
    )

# annotate a few spotlight countries if present
for c in ["RUS", "USA", "CHN", "DEU", "IND", "BRA", "ZAF", "TUR"]:
    if c in reg_profile.index:
        i = reg_profile.index.get_loc(c)
        ax.annotate(c, (X_pca[i, 0], X_pca[i, 1]), fontsize=8, alpha=0.9)

ax.set_title("Country Clusters in PCA Space", fontsize=12, fontweight="bold")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
AM.save_figure(fig, "01_country_clusters")
print("  → output/figures/01_country_clusters.png")

# ── 6.3 Save cluster/vulnerability table ──────────────────────────────────────
cluster_df = reg_profile.reset_index()[["country_code", "cluster", "vulnerability_idx"]]
cluster_df = cluster_df.sort_values(["cluster", "vulnerability_idx"], ascending=[True, False])
AM.save_table(cluster_df, "02_country_clusters_vulnerability")
AM.show_table(cluster_df.head(30), "Country clusters & vulnerability")

# ── 6.4 Governance / institutional heatmap (safe) ────────────────────────────
gov_cols_existing = [c for c in ["governance_mean"] + wgi_present if c in reg_profile.columns]
gov_cols_existing = list(dict.fromkeys(gov_cols_existing))

if len(gov_cols_existing) > 0:
    if "governance_mean" in gov_cols_existing:
        top_gov = reg_profile.sort_values("governance_mean", ascending=False)[gov_cols_existing].head(30)
    else:
        top_gov = reg_profile[gov_cols_existing].dropna(how="all").head(30)

    fig2, ax2 = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        top_gov,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        linewidths=0.4,
        ax=ax2,
        cbar_kws={"label": "Normalised score"}
    )
    ax2.set_title("Top-30 Countries by Governance / Institutional Profile", fontsize=11)
    plt.tight_layout()
    AM.save_figure(fig2, "02_governance_heatmap")
    print("  → output/figures/02_governance_heatmap.png")
else:
    print("  Governance columns not available -> skipping governance heatmap")

# ── 6.5 Vulnerability barplot ─────────────────────────────────────────────────
top_vuln = reg_profile.sort_values("vulnerability_idx", ascending=False).head(20)

fig3, ax3 = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=top_vuln.reset_index(),
    y="country_code",
    x="vulnerability_idx",
    palette="Reds_r",
    ax=ax3
)
ax3.set_title("Top-20 Countries by Vulnerability Index", fontsize=11)
ax3.set_xlabel("Vulnerability index")
ax3.set_ylabel("")
ax3.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
AM.save_figure(fig3, "03_vulnerability_top20")
print("  → output/figures/03_vulnerability_top20.png")

print("✓ EDA / Clustering done")

  7 clusters, PCA var explained: 76.2%
  ⏱  EDA / Clustering: 0.1s
  → output/figures/01_country_clusters.png


,country_code,cluster,vulnerability_idx
34,CHN,0,0.3966
90,ISR,0,0.3738
195,USA,0,0.3738
91,ITA,0,0.2837
142,NZL,0,0.2734
31,CAN,0,0.2644
86,IRL,0,0.2632
62,FRA,0,0.2476
48,DEU,0,0.2332
66,GBR,0,0.2332


  Governance columns not available -> skipping governance heatmap
  → output/figures/03_vulnerability_top20.png
✓ EDA / Clustering done


## Cell 7 · Baseline Forecasting (Holt-Winters ETS, 2025-2050)

In [ ]:
# ── 7.1 Per-country per-feature ETS forecast ─────────────────────────────────
def baseline_forecast(series: pd.Series, horizon: int) -> np.ndarray:
    """Holt-Winters additive trend; fallback to linear regression."""
    vals = series.dropna().values
    if len(vals) < 4:
        return np.full(horizon, vals[-1] if len(vals) > 0 else 0.0)
    try:
        m = ExponentialSmoothing(vals, trend="add", seasonal=None,
                                 initialization_method="estimated")
        return m.fit(optimized=True, disp=False).forecast(horizon)
    except Exception:
        x = np.arange(len(vals))
        slope, intercept, *_ = stats.linregress(x, vals)
        return np.array([intercept + slope*(len(vals)+i) for i in range(1, horizon+1)])

FORECAST_YEARS = list(range(2025, 2025 + CFG.FORECAST_HORIZON))

with Timer("Baseline forecast 2025-2050"):
    baseline_records = []
    for ctry in tqdm(countries, desc="Baseline", leave=False):
        rdf = df_norm[df_norm["country_code"] == ctry].sort_values("year")
        row = {"country_code": ctry}
        for feat in FEAT_COLS:
            if feat in rdf.columns:
                preds = baseline_forecast(rdf.set_index("year")[feat],
                                          CFG.FORECAST_HORIZON)
            else:
                preds = np.zeros(CFG.FORECAST_HORIZON)
            for yr, val in zip(FORECAST_YEARS, preds):
                row[f"{feat}__{yr}"] = val
        baseline_records.append(row)

    # Long format
    bl_long = []
    for rec in baseline_records:
        ctry = rec["country_code"]
        for yr in FORECAST_YEARS:
            row2 = {"country_code": ctry, "year": yr}
            for feat in FEAT_COLS:
                row2[feat] = rec.get(f"{feat}__{yr}", np.nan)
            bl_long.append(row2)
    df_baseline = pd.DataFrame(bl_long)
    print(f"  Baseline forecast rows: {len(df_baseline)}")
    AM.save_table(df_baseline.head(5000), "03_baseline_forecast_sample")

# ── 7.2 Visualise key countries ───────────────────────────────────────────────
SHOWCASE = [c for c in ["RUS","USA","CHN","DEU","BRA","IND","ZAF"]
            if c in countries][:6]
PLOT_FEATS = [f for f in ["gdp_growth","governance_mean",
                           "internet_users","trade_gdp"]
              if f in FEAT_COLS]

plt.style.use(CFG.FIGURE_STYLE)
pal_s = sns.color_palette("husl", len(SHOWCASE))
fig, axes = plt.subplots(len(PLOT_FEATS), 1, figsize=(14, 11))
fig.suptitle("Historical data & ETS baseline forecast 2025-2050",
             fontsize=12, fontweight="bold")

for ai, feat in enumerate(PLOT_FEATS):
    ax = axes[ai]
    for ri, ctry in enumerate(SHOWCASE):
        rh = df_norm[df_norm["country_code"] == ctry].sort_values("year")
        if feat in rh.columns:
            ax.plot(rh["year"], rh[feat], color=pal_s[ri], lw=2.0,
                    label=ctry if ai == 0 else None)
        rb = df_baseline[df_baseline["country_code"] == ctry]
        if feat in rb.columns:
            ax.plot(rb["year"], rb[feat], color=pal_s[ri], lw=1.2, ls="--", alpha=0.7)
    ax.axvline(2024, color="gray", ls=":", lw=1.2, alpha=0.8)
    ax.set_ylabel(feat.replace("_", " "), fontsize=8)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.4)

axes[0].legend(fontsize=7, loc="upper left", ncol=3)
axes[-1].set_xlabel("Year")
plt.tight_layout()
AM.save_figure(fig, "03_baseline_forecasts")
print("  → output/figures/03_baseline_forecasts.png")
print("✓ Baseline forecast done")


  Baseline forecast rows: 5408
  ⏱  Baseline forecast 2025-2050: 10.2s
  → output/figures/03_baseline_forecasts.png
✓ Baseline forecast done


## Cell 8 · World Model: Ensemble GRU with uncertainty estimation

In [ ]:
# ── 8.1 Architecture ─────────────────────────────────────────────────────────
class CountryTransitionNet(nn.Module):
    """Single GRU transition model for one ensemble member.
    Input: state_t | action_t | neighbour_agg_t | crisis_ctx_t
    Output: Δstate (mean + log_std) for next-step probabilistic prediction.
    """
    def __init__(self, state_dim, action_dim, neigh_dim=None,
                 crisis_dim=4, hidden=128, n_layers=2, dropout=0.15):
        super().__init__()
        if neigh_dim is None:
            neigh_dim = state_dim
        input_dim = state_dim + action_dim + neigh_dim + crisis_dim
        self.gru = nn.GRU(input_dim, hidden, num_layers=n_layers,
                          batch_first=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc_mean = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.SiLU(),
            nn.Linear(hidden, state_dim),
        )
        self.fc_std = nn.Linear(hidden, state_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, s, a, neigh, crisis, hid=None):
        # s,a,neigh,crisis: (B, dim)
        x = torch.cat([s, a, neigh, crisis], dim=-1).unsqueeze(1)  # (B,1,D)
        out, h = self.gru(x, hid)
        out = self.drop(out.squeeze(1))   # (B, hidden)
        delta   = self.fc_mean(out)
        log_std = self.fc_std(out).clamp(-4, 2)
        return delta, log_std, h


class EnsembleWorldModel(nn.Module):
    """Ensemble of K GRU models; returns mean prediction + epistemic uncertainty."""
    def __init__(self, state_dim, action_dim, ensemble_size=3, **kw):
        super().__init__()
        self.models = nn.ModuleList([
            CountryTransitionNet(state_dim, action_dim, **kw)
            for _ in range(ensemble_size)
        ])
        self.K = ensemble_size

    def forward(self, s, a, neigh, crisis):
        preds = [m(s, a, neigh, crisis)[0] for m in self.models]
        preds_t = torch.stack(preds, dim=0)  # (K, B, S)
        return preds_t.mean(0), preds_t.std(0)  # mean, uncertainty

    def single_forward(self, k, s, a, neigh, crisis, hid=None):
        return self.models[k](s, a, neigh, crisis, hid)


SDIM = CFG.STATE_DIM
ADIM = CFG.ACTION_DIM

world_model = EnsembleWorldModel(
    state_dim=SDIM, action_dim=ADIM,
    neigh_dim=SDIM, crisis_dim=4,
    ensemble_size=CFG.WM_ENSEMBLE_SIZE,
    hidden=CFG.WM_HIDDEN_DIM,
    n_layers=CFG.WM_N_LAYERS,
    dropout=CFG.WM_DROPOUT,
).to(DEVICE)

total_p = sum(p.numel() for p in world_model.parameters())
print(f"  EnsembleWorldModel params: {total_p:,}  "
      f"({CFG.WM_ENSEMBLE_SIZE} × {total_p//CFG.WM_ENSEMBLE_SIZE:,})")
print("✓ World Model architecture ready")


  EnsembleWorldModel params: 625,746  (3 × 208,582)
✓ World Model architecture ready


## Cell 9 · World Model: dataset construction & training

In [9]:
# ── 9.1 Crisis year context dictionary ───────────────────────────────────────
# 4-vector: (gdp_shock, mortality_shock, migration_shock, energy_shock)
CRISIS_YEAR_CTX: Dict[int, Tuple] = {
    1998: (-0.10, 0.05, -0.20, -0.10),  # Asian financial crisis tail
    2001: (-0.08, 0.02, -0.15, -0.05),  # Dot-com + 9/11
    2008: (-0.25, 0.10, -0.40, -0.30),  # Global financial crisis
    2009: (-0.20, 0.08, -0.35, -0.25),  # GFC continued
    2020: (-0.15, 0.25, -0.60, -0.20),  # COVID-19
    2021: ( 0.05, 0.10, -0.30, -0.10),  # Recovery + supply chain
    2022: (-0.05, 0.03, -0.20, -0.45),  # Energy/inflation shock
}

def build_wm_dataset(df_split, adj_mat, ctry_idx, state_cols,
                     action_dim, ctry_list):
    """Build (St, At, Nt, Ct, St+1) tensors from panel data."""
    records = []
    years = sorted(df_split["year"].unique())
    for yi in range(len(years) - 1):
        yr, yr1 = years[yi], years[yi + 1]
        dfy  = df_split[df_split["year"] == yr ].set_index("country_code")
        dfy1 = df_split[df_split["year"] == yr1].set_index("country_code")
        c_ctx = torch.tensor(CRISIS_YEAR_CTX.get(yr, (0,0,0,0)),
                             dtype=torch.float32)
        for ctry in ctry_list:
            if ctry not in dfy.index or ctry not in dfy1.index:
                continue
            ri = ctry_idx.get(ctry, 0)
            st  = torch.tensor(dfy.loc[ctry, state_cols].values.astype(float),
                               dtype=torch.float32)
            st1 = torch.tensor(dfy1.loc[ctry, state_cols].values.astype(float),
                               dtype=torch.float32)
            # neighbour aggregate
            ns = torch.zeros(len(state_cols), dtype=torch.float32)
            for c2 in ctry_list:
                w = adj_mat[ri, ctry_idx.get(c2, 0)]
                if w > 0 and c2 in dfy.index:
                    ns += w * torch.tensor(dfy.loc[c2, state_cols].values.astype(float),
                                           dtype=torch.float32)
            at = torch.zeros(action_dim, dtype=torch.float32)
            st  = torch.nan_to_num(st)
            st1 = torch.nan_to_num(st1)
            ns  = torch.nan_to_num(ns)
            records.append((st, at, ns, c_ctx, st1))
    if not records:
        z = lambda d: torch.zeros(1, d)
        return z(SDIM), z(ADIM), z(SDIM), z(4), z(SDIM)
    S, A, N, C, S1 = zip(*records)
    return (torch.stack(S), torch.stack(A), torch.stack(N),
            torch.stack([c.expand(4) for c in C]), torch.stack(S1))

print("Building WM datasets...")
with Timer("Build WM datasets"):
    tr_S,tr_A,tr_N,tr_C,tr_S1 = build_wm_dataset(
        df_train, ADJ_NORM, CTRY_IDX, FEAT_COLS, ADIM, countries)
    va_S,va_A,va_N,va_C,va_S1 = build_wm_dataset(
        df_val,   ADJ_NORM, CTRY_IDX, FEAT_COLS, ADIM, countries)
    print(f"  Train WM: {tr_S.shape[0]}   Val WM: {va_S.shape[0]}")

# ── 9.2 Training ─────────────────────────────────────────────────────────────
def train_world_model(wm, tr_data, va_data, cfg):
    tr_S,tr_A,tr_N,tr_C,tr_S1 = [t.to(DEVICE) for t in tr_data]
    va_S,va_A,va_N,va_C,va_S1 = [t.to(DEVICE) for t in va_data]
    N_tr, B = tr_S.shape[0], cfg.WM_BATCH_SIZE
    hist = {"train": [], "val": []}

    for k, model_k in enumerate(wm.models):
        opt   = Adam(model_k.parameters(), lr=cfg.WM_LR, weight_decay=1e-5)
        sched = CosineAnnealingLR(opt, T_max=cfg.WM_EPOCHS, eta_min=1e-6)
        es_wm = EarlyStopping(patience=cfg.WM_PATIENCE, mode="min", restore_best=True)

        for ep in range(cfg.WM_EPOCHS):
            model_k.train()
            idx_list = torch.randperm(N_tr, device=DEVICE)
            ep_loss, nb = 0.0, 0
            for s in range(0, N_tr, B):
                idx = idx_list[s:s+B]
                if len(idx) < 2:
                    continue
                d_pred, ls, _ = model_k(tr_S[idx], tr_A[idx],
                                         tr_N[idx], tr_C[idx])
                d_true = tr_S1[idx] - tr_S[idx]
                var    = torch.exp(2*ls) + 1e-8
                loss   = ((d_true - d_pred)**2 / (2*var) + ls).mean()
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model_k.parameters(), 1.0)
                opt.step()
                ep_loss += loss.item(); nb += 1
            sched.step()
            tr_l = ep_loss / max(nb, 1)

            model_k.eval()
            with torch.no_grad():
                dp, ls2, _ = model_k(va_S, va_A, va_N, va_C)
                dt = va_S1 - va_S
                var2 = torch.exp(2*ls2) + 1e-8
                val_l = ((dt - dp)**2 / (2*var2) + ls2).mean().item()

            if k == 0:
                hist["train"].append(tr_l)
                hist["val"].append(val_l)

            if es_wm.step(val_l, model_k):
                if k == 0:
                    print(f"    WM{k} early stop ep{ep+1}  val={val_l:.4f}")
                break
        if k == 0:
            print(f"    WM{k} best_val={es_wm.best_score:.4f}")
    return hist

with Timer("Train World Model"):
    wm_hist = train_world_model(
        world_model,
        (tr_S, tr_A, tr_N, tr_C, tr_S1),
        (va_S, va_A, va_N, va_C, va_S1),
        CFG)
    torch.save(world_model.state_dict(), "output/checkpoints/world_model.pt")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wm_hist["train"], label="Train NLL", color="steelblue")
ax.plot(wm_hist["val"],   label="Val NLL",   color="coral")
ax.set_xlabel("Epoch"); ax.set_ylabel("NLL Loss")
ax.set_title("World Model – Learning Curve")
ax.legend(); ax.grid(True, alpha=0.4)
AM.save_figure(fig, "04_wm_learning_curve")
print("  → output/figures/04_wm_learning_curve.png")
print("✓ World Model trained")


Building WM datasets...
  Train WM: 2912   Val WM: 1040
  ⏱  Build WM datasets: 392.8s
    WM0 early stop ep119  val=-0.9530
    WM0 best_val=-1.1616
  ⏱  Train World Model: 24.6s
  → output/figures/04_wm_learning_curve.png
✓ World Model trained


## Cell 10 · World Model Validation: MAE & Uncertainty

In [10]:
# ── 10.1 MAE per feature on validation set ───────────────────────────────────
world_model.eval()
with torch.no_grad():
    va_Sd, va_Ad, va_Nd, va_Cd = [t.to(DEVICE) for t in (va_S, va_A, va_N, va_C)]
    m_pred, uncert = world_model(va_Sd, va_Ad, va_Nd, va_Cd)
    next_pred = (va_Sd + m_pred).cpu().numpy()
    next_true = va_S1.numpy()
    unc_np    = uncert.cpu().numpy()

mae_vals, unc_vals = {}, {}
n_feats = min(len(FEAT_COLS), next_pred.shape[1])
for fi, feat in enumerate(FEAT_COLS[:n_feats]):
    if fi < next_pred.shape[1]:
        mae_vals[feat] = mean_absolute_error(next_true[:, fi], next_pred[:, fi])
        unc_vals[feat] = unc_np[:, fi].mean()

wm_mae_df = pd.DataFrame({
    "feature":        list(mae_vals.keys()),
    "MAE":            list(mae_vals.values()),
    "avg_uncertainty": list(unc_vals.values()),
}).round(4)
print("World Model MAE (val set, normalised units):")
print(wm_mae_df.to_string(index=False))
AM.save_table(wm_mae_df, "04_world_model_mae")
AM.show_table(wm_mae_df, "World Model MAE by feature")

# ── 10.2 Scatter pred vs true (top 8 features by MAE) ───────────────────────
feat_list = wm_mae_df.sort_values("MAE").head(8)["feature"].tolist()
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
fig.suptitle("World Model: Predicted vs True (val set)", fontsize=12, fontweight="bold")
for i, (feat, ax) in enumerate(zip(feat_list, axes.flat)):
    fi = FEAT_COLS.index(feat) if feat in FEAT_COLS else -1
    if fi < 0 or fi >= next_pred.shape[1]:
        continue
    ax.scatter(next_true[:, fi], next_pred[:, fi], alpha=0.25, s=8, c="steelblue")
    lims = [min(next_true[:,fi].min(), next_pred[:,fi].min()),
            max(next_true[:,fi].max(), next_pred[:,fi].max())]
    ax.plot(lims, lims, "r--", lw=1.2)
    ax.set_title(feat.replace("_"," "), fontsize=7)
    ax.set_xlabel("True", fontsize=7); ax.set_ylabel("Pred", fontsize=7)
    ax.text(0.05, 0.92, f"MAE={mae_vals[feat]:.3f}",
            transform=ax.transAxes, fontsize=7, color="crimson")
    ax.tick_params(labelsize=6); ax.grid(True, alpha=0.3)
plt.tight_layout()
AM.save_figure(fig, "05_wm_validation")
print("  → output/figures/05_wm_validation.png")
print("✓ World Model validation done")


World Model MAE (val set, normalised units):
            feature    MAE  avg_uncertainty
         gdp_growth 0.6056           0.0213
         gdp_pc_usd 0.0386           0.0161
      gdp_pc_growth 0.6265           0.0366
          inflation 0.4976           0.0214
       unemployment 0.0740           0.0249
 gross_capital_form 0.2643           0.0220
   govt_consumption 0.1013           0.0158
        exports_gdp 0.1033           0.0182
        imports_gdp 0.1163           0.0142
          trade_gdp 0.1082           0.0167
            fdi_gdp 1.8305           0.0406
  manufacturing_gdp 0.0593           0.0194
  nat_resources_gdp 0.1685           0.0148
             rd_gdp 0.0816           0.0126
     researchers_pm 0.0336           0.0080
   journal_articles 0.0337           0.0086
  patents_residents 0.0514           0.0102
     hitech_exports 0.1787           0.0112
     internet_users 0.0393           0.0126
        mobile_subs 0.0904           0.0122
          broadband 0.0663     

,feature,MAE,avg_uncertainty
0,gdp_growth,0.6056,0.0213
1,gdp_pc_usd,0.0386,0.0161
2,gdp_pc_growth,0.6265,0.0366
3,inflation,0.4976,0.0214
4,unemployment,0.0740,0.0249
5,gross_capital_form,0.2643,0.0220
6,govt_consumption,0.1013,0.0158
7,exports_gdp,0.1033,0.0182
8,imports_gdp,0.1163,0.0142
9,trade_gdp,0.1082,0.0167


  → output/figures/05_wm_validation.png
✓ World Model validation done


## Cell 11 · MARL Environment: WorldPoliticsEnv

In [11]:
# ── 11.1 Crisis scenarios ─────────────────────────────────────────────────────
# Each scenario: (gdp_shock, mortality_shock, migration_shock, energy_shock)
# Named for scientific clarity; avoids politically evaluative framing.
CRISIS_SCENARIOS = [
    {"id":1,  "name":"Global Pandemic",        "b":-0.15,"d": 0.25,"m":-0.60,"e":-0.20},
    {"id":2,  "name":"Sanctions Regime",       "b":-0.20,"d": 0.05,"m":-0.50,"e":-0.35},
    {"id":3,  "name":"Energy Price Shock",     "b":-0.18,"d": 0.10,"m":-0.30,"e":-0.60},
    {"id":4,  "name":"Financial Crisis",       "b":-0.25,"d": 0.08,"m":-0.40,"e":-0.30},
    {"id":5,  "name":"Climate Catastrophe",   "b":-0.10,"d": 0.20,"m":-0.50,"e":-0.25},
    {"id":6,  "name":"Tech Disruption",        "b":-0.08,"d": 0.03,"m":-0.20,"e":-0.15},
    {"id":7,  "name":"Food Security Crisis",  "b":-0.12,"d": 0.30,"m":-0.45,"e":-0.20},
    {"id":8,  "name":"Institutional Crisis",  "b":-0.15,"d": 0.05,"m":-0.35,"e":-0.20},
    {"id":9,  "name":"Infrastructure Shock",  "b":-0.20,"d": 0.15,"m":-0.40,"e":-0.40},
    {"id":10, "name":"Trade Fragmentation",   "b":-0.18,"d": 0.02,"m":-0.25,"e":-0.25},
]

# ── 11.2 Environment ─────────────────────────────────────────────────────────
class WorldPoliticsEnv:
    """Multi-agent RL environment: countries as agents.

    State:  FEAT_COLS (WDI + WGI), dim = STATE_DIM
    Action: 14 political-economic instruments in [0,1], sum ≤ BUDGET_LIMIT
    Reward: weighted multi-objective: growth, governance, social, ecology

    The World Model (EnsembleWorldModel) is used as the simulator.
    Crisis shocks are injected stochastically, scaled by country vulnerability.
    """

    def __init__(self, world_model, initial_states, adj_mat,
                 vuln_map, country_list, cfg, feat_cols):
        self.wm       = world_model
        self.adj      = torch.tensor(adj_mat, dtype=torch.float32, device=DEVICE)
        self.n        = len(country_list)
        self.sdim     = cfg.STATE_DIM
        self.adim     = cfg.ACTION_DIM
        self.cfg      = cfg
        self.cols     = feat_cols
        self.countries = country_list
        self.vuln     = np.array([vuln_map.get(c, 0.5) for c in country_list],
                                 dtype=np.float32)
        self.ep_len   = cfg.MADDPG_EPISODE_LEN
        self._init_states = initial_states.copy()
        self.states   = initial_states.copy()
        self.step_cnt = 0
        self.crisis   = None
        self.crisis_t = 0

        # Feature indices for reward calculation
        fi = {f: i for i, f in enumerate(feat_cols)}
        self._i = fi  # full map
        self.i_gdp_g   = fi.get("gdp_growth",         0)
        self.i_unmp    = fi.get("unemployment",        4)
        self.i_inf_m   = fi.get("infant_mortality",    22)
        self.i_gini    = fi.get("gini_index",           29)
        self.i_trade   = fi.get("trade_gdp",           10)
        self.i_renew   = fi.get("renewable_energy",    32)
        self.i_internet= fi.get("internet_users",      18)
        # WGI governance indices
        self.i_gov     = [fi.get(g, -1)
                          for g in ["va_score","pv_score","ge_score",
                                    "rq_score","rl_score","cc_score"]
                          if fi.get(g, -1) >= 0]

    def reset(self, init=None):
        self.states   = (init if init is not None else self._init_states).copy()
        self.step_cnt = 0
        self.crisis   = None
        self.crisis_t = 0
        return self.states.copy()

    def _crisis_ctx(self):
        if self.crisis_t > 0:
            self.crisis_t -= 1
            c = self.crisis
            return np.array([c["b"],c["d"],c["m"],c["e"]], dtype=np.float32), c
        r = np.random.random()
        if r < self.cfg.TAIL_RISK_PROB:
            c = CRISIS_SCENARIOS[np.random.randint(6, 10)]
        elif r < self.cfg.CRISIS_INJECT_PROB:
            c = CRISIS_SCENARIOS[np.random.randint(0, 6)]
        else:
            return np.zeros(4, dtype=np.float32), None
        self.crisis   = c
        self.crisis_t = np.random.randint(1, 4)
        return np.array([c["b"],c["d"],c["m"],c["e"]], dtype=np.float32), c

    def step(self, actions):
        # Clip and normalise actions to budget
        actions = np.clip(actions, 0, 1)
        budget  = actions.sum(axis=1, keepdims=True)
        scale   = np.where(budget > self.cfg.BUDGET_LIMIT,
                           self.cfg.BUDGET_LIMIT / (budget + 1e-8), 1.0)
        actions = actions * scale

        ctx_np, c_obj = self._crisis_ctx()
        c_batch = (np.outer(self.vuln, ctx_np)
                   if c_obj is not None
                   else np.zeros((self.n, 4), dtype=np.float32))

        st  = torch.tensor(self.states, dtype=torch.float32, device=DEVICE)
        at  = torch.tensor(actions,     dtype=torch.float32, device=DEVICE)
        nt  = self.adj @ st        # neighbour aggregate (N, S)
        ct  = torch.tensor(c_batch, dtype=torch.float32, device=DEVICE)

        self.wm.eval()
        with torch.no_grad():
            delta_mean, uncert = self.wm(st, at, nt, ct)
            noise  = torch.randn_like(delta_mean) * uncert * 0.05
            next_s = (st + delta_mean + noise).cpu().numpy()

        rewards = self._reward(self.states, next_s, actions, c_obj)
        self.states   = next_s
        self.step_cnt += 1
        done = (self.step_cnt >= self.ep_len)
        info = {"crisis": c_obj["name"] if c_obj else None,
                "unc":    uncert.mean().item()}
        return next_s, rewards, done, info

    def _reward(self, sp, sn, a, c_obj):
        W   = self.cfg.REWARD_WEIGHTS
        t   = np.tanh
        def g(x, i): return x[:, i] if i < x.shape[1] else np.zeros(x.shape[0])

        # GDP growth improvement
        r = W["gdp_growth"] * t((g(sn,self.i_gdp_g) - g(sp,self.i_gdp_g)) * 5)
        # Governance improvement (mean of available WGI indices)
        if self.i_gov:
            gov_delta = np.mean([t((g(sn,i)-g(sp,i))*3) for i in self.i_gov], axis=0)
            r += W["governance_mean"] * gov_delta
        # Trade openness
        r += W["trade_gdp"] * t((g(sn,self.i_trade) - g(sp,self.i_trade))*3)
        # Unemployment penalty (↓ is good)
        r += W["unemployment"] * t(-(g(sn,self.i_unmp) - g(sp,self.i_unmp))*3)
        # Infant mortality penalty
        r += W["infant_mortality"] * t(-(g(sn,self.i_inf_m) - g(sp,self.i_inf_m))*3)
        # Gini penalty
        r += W["gini_index"] * t(-(g(sn,self.i_gini) - g(sp,self.i_gini))*3)
        # Renewable energy
        r += W["renewable_energy"] * t((g(sn,self.i_renew) - g(sp,self.i_renew))*3)
        # Internet access
        r += W["internet_users"] * t((g(sn,self.i_internet) - g(sp,self.i_internet))*3)
        # Mild penalty for excessive military buildup (action A02)
        r += W["military_budget"] * (-np.clip(a[:, 1] - 0.3, 0, None))
        # Budget penalty
        r += W["budget_penalty"] * (-np.maximum(0, a.sum(1) - self.cfg.BUDGET_LIMIT))
        return r.astype(np.float32)


def build_init_states(df_split, ctry_list, state_cols, year=None):
    y   = year if year else df_split["year"].max()
    dfy = df_split[df_split["year"] == y].set_index("country_code")
    st  = np.zeros((len(ctry_list), len(state_cols)), dtype=np.float32)
    for i, c in enumerate(ctry_list):
        if c in dfy.index:
            st[i] = np.nan_to_num(dfy.loc[c, state_cols].values.astype(float))
    return st

init_tr = build_init_states(df_train, countries, FEAT_COLS)
init_te = build_init_states(df_test  if len(df_test) > 0 else df_norm,
                            countries, FEAT_COLS)

env_train = WorldPoliticsEnv(world_model, init_tr, ADJ_NORM,
                             VULN_MAP, countries, CFG, FEAT_COLS)
env_test  = WorldPoliticsEnv(world_model, init_te, ADJ_NORM,
                             VULN_MAP, countries, CFG, FEAT_COLS)

print(f"  WorldPoliticsEnv: {env_train.n} countries")
print(f"  State dim: {env_train.sdim}   Action dim: {env_train.adim}")
print(f"  Episode length: {env_train.ep_len}   Budget limit: {CFG.BUDGET_LIMIT}")
print("✓ Environment ready")


  WorldPoliticsEnv: 208 countries
  State dim: 35   Action dim: 14
  Episode length: 18   Budget limit: 4.0
✓ Environment ready


## Cell 12 · MARL Networks: SharedActor, TwinCritic, ReplayBuffer

In [12]:
# ── 12.1 Shared Actor (parameter sharing across all country-agents) ───────────
class SharedActor(nn.Module):
    """Maps state → action ∈ [0,1]^A (Sigmoid output)."""
    def __init__(self, sdim, adim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(sdim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, adim), nn.Sigmoid(),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=0.01)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, s):
        return self.net(s)


# ── 12.2 Twin Critic (for MATD3, reduces overestimation) ─────────────────────
class TwinCritic(nn.Module):
    """Q1, Q2 heads to reduce overestimation bias (TD3-style)."""
    def __init__(self, sdim, adim, hidden=256):
        super().__init__()
        ind = sdim * 2 + adim   # state + neighbour_agg + action
        def mlp():
            return nn.Sequential(
                nn.Linear(ind, hidden), nn.LayerNorm(hidden), nn.GELU(),
                nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
                nn.Linear(hidden, 1),
            )
        self.q1 = mlp(); self.q2 = mlp()

    def forward(self, s, a, ns):
        x = torch.cat([s, a, ns], dim=-1)
        return self.q1(x), self.q2(x)

    def q1_only(self, s, a, ns):
        return self.q1(torch.cat([s, a, ns], dim=-1))


# MADDPG uses single critic
class SharedCritic(nn.Module):
    def __init__(self, sdim, adim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(sdim*2+adim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, s, a, ns):
        return self.net(torch.cat([s, a, ns], dim=-1))


# ── 12.3 Replay Buffer ────────────────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, cap, n, sdim, adim):
        self.cap, self.n, self.ptr, self.size = cap, n, 0, 0
        self.S  = np.zeros((cap, n, sdim), dtype=np.float32)
        self.A  = np.zeros((cap, n, adim), dtype=np.float32)
        self.R  = np.zeros((cap, n),       dtype=np.float32)
        self.S1 = np.zeros((cap, n, sdim), dtype=np.float32)
        self.D  = np.zeros((cap,),         dtype=np.float32)

    def push(self, s, a, r, s1, done):
        self.S[self.ptr]  = s;  self.A[self.ptr] = a
        self.R[self.ptr]  = r;  self.S1[self.ptr]= s1
        self.D[self.ptr]  = float(done)
        self.ptr  = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, B):
        idx = np.random.randint(0, self.size, B)
        to  = lambda x: torch.tensor(x[idx], device=DEVICE)
        return to(self.S), to(self.A), to(self.R), to(self.S1), to(self.D)

    def __len__(self): return self.size


# ── 12.4 Helper functions ─────────────────────────────────────────────────────
def ou_noise(a, sigma=0.15):
    return (a + sigma * torch.randn_like(a)).clamp(0, 1)

def neigh_agg(states, adj):
    """Batch neighbour aggregation: (B,N,S) × (N,N) → (B,N,S)."""
    return torch.bmm(adj.unsqueeze(0).expand(states.shape[0],-1,-1), states)

print("✓ Networks & ReplayBuffer defined")


✓ Networks & ReplayBuffer defined


## Cell 13 · MADDPG Training

In [13]:
def train_maddpg(env, cfg, logger):
    """MADDPG with shared actor/critic and parameter sharing."""
    N, S, A = env.n, env.sdim, env.adim
    actor  = SharedActor(S, A, cfg.MADDPG_HIDDEN).to(DEVICE)
    critic = SharedCritic(S, A, cfg.MADDPG_HIDDEN).to(DEVICE)
    ta, tc = copy.deepcopy(actor).to(DEVICE), copy.deepcopy(critic).to(DEVICE)
    for p in list(ta.parameters()) + list(tc.parameters()):
        p.requires_grad_(False)

    opt_a = Adam(actor.parameters(),  lr=cfg.MADDPG_LR_ACTOR,  weight_decay=1e-5)
    opt_c = Adam(critic.parameters(), lr=cfg.MADDPG_LR_CRITIC, weight_decay=1e-5)
    buf   = ReplayBuffer(cfg.MADDPG_BUFFER_SIZE, N, S, A)
    es    = EarlyStopping(patience=cfg.RL_PATIENCE, mode="max", restore_best=True)
    adj_t = ADJ_TENSOR
    sigma = 0.20; total_steps = 0; best_rew = -np.inf; loss_cv = 0.0

    for ep in trange(cfg.MADDPG_EPISODES, desc="MADDPG", leave=True):
        states = env.reset(init_tr.copy()); ep_rew = 0.0
        for _ in range(cfg.MADDPG_EPISODE_LEN):
            st = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts = (ou_noise(actor(st), sigma)
                        if total_steps >= cfg.MADDPG_WARMUP
                        else torch.rand(N, A, device=DEVICE))
            nxt, rws, done, _ = env.step(acts.cpu().numpy())
            buf.push(states, acts.cpu().numpy(), rws, nxt, done)
            ep_rew = rws.mean(); states = nxt; total_steps += 1

            if len(buf) < cfg.MADDPG_BATCH_SIZE or total_steps < cfg.MADDPG_WARMUP:
                if done: break
                continue

            Sb,Ab,Rb,S1b,Db = buf.sample(cfg.MADDPG_BATCH_SIZE)
            ns  = neigh_agg(Sb,  adj_t); ns1 = neigh_agg(S1b, adj_t)
            B2  = Sb.shape[0]

            with torch.no_grad():
                A1 = ta(S1b.view(-1,S))
                qt = tc(S1b.view(-1,S), A1, ns1.view(-1,S)).view(B2,N)
                td = Rb + cfg.MADDPG_GAMMA * qt * (1 - Db.unsqueeze(1))

            qp = critic(Sb.view(-1,S), Ab.view(-1,A), ns.view(-1,S)).view(B2,N)
            lc = F.mse_loss(qp, td.detach())
            opt_c.zero_grad(); lc.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), 1.0); opt_c.step()
            loss_cv = lc.item()

            actor.train()
            ac = actor(Sb.view(-1,S))
            la = -critic(Sb.view(-1,S), ac, ns.view(-1,S).detach()).mean()
            opt_a.zero_grad(); la.backward()
            nn.utils.clip_grad_norm_(actor.parameters(), 1.0); opt_a.step()

            for p, tp in zip(actor.parameters(), ta.parameters()):
                tp.data.copy_(cfg.MADDPG_TAU*p.data + (1-cfg.MADDPG_TAU)*tp.data)
            for p, tp in zip(critic.parameters(), tc.parameters()):
                tp.data.copy_(cfg.MADDPG_TAU*p.data + (1-cfg.MADDPG_TAU)*tp.data)
            if done: break

        sigma = max(0.02, sigma * 0.995)
        logger.log(ep, {"reward": ep_rew, "sigma": sigma, "loss_c": loss_cv})
        if ep_rew > best_rew:
            best_rew = ep_rew
            torch.save(actor.state_dict(), "output/checkpoints/maddpg_actor_best.pt")
        if ep % cfg.MADDPG_LOG_INTERVAL == 0:
            tqdm.write(f"  MADDPG ep{ep:4d} rew={ep_rew:.4f} σ={sigma:.3f}")
        if es.step(ep_rew) and ep > 150:
            tqdm.write(f"  MADDPG early stop ep{ep}"); break

    logger.save()
    actor.load_state_dict(torch.load("output/checkpoints/maddpg_actor_best.pt",
                                     map_location=DEVICE))
    return actor, critic

with Timer("MADDPG"):
    maddpg_log   = ExperimentLogger("MADDPG")
    maddpg_actor, maddpg_critic = train_maddpg(env_train, CFG, maddpg_log)
print("✓ MADDPG done")


MADDPG:   0%|          | 2/1800 [00:00<04:06,  7.28it/s]

  MADDPG ep   0 rew=0.0074 σ=0.199


MADDPG:   3%|▎         | 51/1800 [00:11<25:29,  1.14it/s]

  MADDPG ep  50 rew=0.0102 σ=0.155


MADDPG:   6%|▌         | 101/1800 [01:11<34:56,  1.23s/it]

...

  MADDPG ep 250 rew=0.0105 σ=0.057


MADDPG:  17%|█▋        | 301/1800 [05:12<30:13,  1.21s/it]

  MADDPG ep 300 rew=0.0108 σ=0.044


MADDPG:  17%|█▋        | 305/1800 [05:18<25:59,  1.04s/it]

  MADDPG early stop ep305
  ⏱  MADDPG: 318.3s
✓ MADDPG done


## Cell 14 · MATD3 Training (Twin Critic + Policy Delay)

In [14]:
def train_matd3(env, cfg, logger):
    """MATD3: Twin critic + delayed policy updates + target policy smoothing."""
    N, S, A = env.n, env.sdim, env.adim
    actor  = SharedActor(S, A, cfg.MATD3_HIDDEN).to(DEVICE)
    critic = TwinCritic(S, A, cfg.MATD3_HIDDEN).to(DEVICE)
    ta, tc = copy.deepcopy(actor).to(DEVICE), copy.deepcopy(critic).to(DEVICE)
    for p in list(ta.parameters()) + list(tc.parameters()):
        p.requires_grad_(False)

    opt_a  = Adam(actor.parameters(),  lr=cfg.MATD3_LR_ACTOR,  weight_decay=1e-5)
    opt_c  = Adam(critic.parameters(), lr=cfg.MATD3_LR_CRITIC, weight_decay=1e-5)
    buf    = ReplayBuffer(cfg.MADDPG_BUFFER_SIZE, N, S, A)
    es     = EarlyStopping(patience=cfg.RL_PATIENCE, mode="max", restore_best=True)
    adj_t  = ADJ_TENSOR
    sigma  = 0.20; total_steps = 0; a_step = 0
    best_rew = -np.inf; lcv = 0.0; lav = 0.0

    for ep in trange(cfg.MATD3_EPISODES, desc="MATD3", leave=True):
        states = env.reset(init_tr.copy()); ep_rew = 0.0
        for _ in range(cfg.MADDPG_EPISODE_LEN):
            st = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts = (ou_noise(actor(st), sigma)
                        if total_steps >= cfg.MADDPG_WARMUP
                        else torch.rand(N, A, device=DEVICE))
            nxt, rws, done, _ = env.step(acts.cpu().numpy())
            buf.push(states, acts.cpu().numpy(), rws, nxt, done)
            ep_rew = rws.mean(); states = nxt; total_steps += 1

            if len(buf) < cfg.MADDPG_BATCH_SIZE or total_steps < cfg.MADDPG_WARMUP:
                if done: break
                continue

            Sb,Ab,Rb,S1b,Db = buf.sample(cfg.MADDPG_BATCH_SIZE)
            ns  = neigh_agg(Sb,  adj_t); ns1 = neigh_agg(S1b, adj_t)
            B2  = Sb.shape[0]; a_step += 1

            with torch.no_grad():
                A1    = ta(S1b.view(-1,S))
                noise = torch.randn_like(A1)*cfg.MATD3_TARGET_NOISE
                noise = noise.clamp(-cfg.MATD3_NOISE_CLIP, cfg.MATD3_NOISE_CLIP)
                A1    = (A1 + noise).clamp(0, 1)
                q1t, q2t = tc(S1b.view(-1,S), A1, ns1.view(-1,S))
                td = Rb + cfg.MATD3_GAMMA * (
                    torch.min(q1t, q2t).view(B2, N) * (1 - Db.unsqueeze(1)))

            q1p, q2p = critic(Sb.view(-1,S), Ab.view(-1,A), ns.view(-1,S))
            lc = (F.mse_loss(q1p.view(B2,N), td.detach()) +
                  F.mse_loss(q2p.view(B2,N), td.detach()))
            opt_c.zero_grad(); lc.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
            opt_c.step(); lcv = lc.item()

            if a_step % cfg.MATD3_POLICY_DELAY == 0:
                actor.train()
                ac = actor(Sb.view(-1,S))
                la = -critic.q1_only(Sb.view(-1,S), ac, ns.view(-1,S).detach()).mean()
                opt_a.zero_grad(); la.backward()
                nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
                opt_a.step(); lav = la.item()
                for p, tp in zip(actor.parameters(), ta.parameters()):
                    tp.data.copy_(cfg.MATD3_TAU*p.data + (1-cfg.MATD3_TAU)*tp.data)
                for p, tp in zip(critic.parameters(), tc.parameters()):
                    tp.data.copy_(cfg.MATD3_TAU*p.data + (1-cfg.MATD3_TAU)*tp.data)
            if done: break

        sigma = max(0.02, sigma * 0.995)
        logger.log(ep, {"reward": ep_rew, "loss_c": lcv, "loss_a": lav})
        if ep_rew > best_rew:
            best_rew = ep_rew
            torch.save(actor.state_dict(), "output/checkpoints/matd3_actor_best.pt")
        if ep % cfg.MATD3_LOG_INTERVAL == 0:
            tqdm.write(f"  MATD3 ep{ep:4d} rew={ep_rew:.4f}")
        if es.step(ep_rew) and ep > 150:
            tqdm.write(f"  MATD3 early stop ep{ep}"); break

    logger.save()
    actor.load_state_dict(torch.load("output/checkpoints/matd3_actor_best.pt",
                                     map_location=DEVICE))
    return actor, critic

with Timer("MATD3"):
    matd3_log   = ExperimentLogger("MATD3")
    matd3_actor, matd3_critic = train_matd3(env_train, CFG, matd3_log)
print("✓ MATD3 done")


MATD3:   0%|          | 0/1800 [00:00<?, ?it/s]

  MATD3 ep   0 rew=0.0076


MATD3:   3%|▎         | 51/1800 [00:13<34:50,  1.20s/it]

  MATD3 ep  50 rew=0.0105


MATD3:   6%|▌         | 101/1800 [01:34<45:58,  1.62s/it]

...

  MATD3 ep 500 rew=0.0110


MATD3:  31%|███       | 551/1800 [13:41<33:34,  1.61s/it]

  MATD3 ep 550 rew=0.0109


MATD3:  32%|███▏      | 576/1800 [14:23<30:34,  1.50s/it]

  MATD3 early stop ep576
  ⏱  MATD3: 863.2s
✓ MATD3 done


## Cell 15 · Evo-MATD3-SR: Evolutionary Self-Referential Action-Bias Fine-tuning

In [15]:
# ── 15.1 Bias Wrapper ─────────────────────────────────────────────────────────
class BiasWrapper(nn.Module):
    """Wraps a frozen base-actor with a learnable action-bias vector b.
    output = clamp(base_actor(s) + b, 0, 1)
    Bias is optimised evolutionarily, not by gradient.
    """
    def __init__(self, base_actor: nn.Module, action_dim: int, init_bias=0.0):
        super().__init__()
        self.base_actor = base_actor
        self.bias = nn.Parameter(
            torch.ones(action_dim) * init_bias, requires_grad=False)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        dev = s.device
        self.base_actor.to(dev)
        if self.bias.device != dev:
            self.bias.data = self.bias.data.to(dev)
        with torch.no_grad():
            a = self.base_actor(s)
        b = self.bias.view(1, -1).expand_as(a)
        return torch.clamp(a + b, 0.0, 1.0)


# ── 15.2 Fitness evaluation ──────────────────────────────────────────────────
def evaluate_bias_candidate(bias_vec, base_actor, env, init_states,
                             n_episodes=5, max_len=None):
    """Rollout episodes with given bias; return mean reward."""
    if max_len is None:
        max_len = env.ep_len
    actor = BiasWrapper(base_actor, env.adim)
    with torch.no_grad():
        actor.bias.data.copy_(
            torch.tensor(bias_vec, dtype=torch.float32, device=DEVICE))
    actor.to(DEVICE)
    rewards = []
    for _ in range(n_episodes):
        states  = env.reset(init_states.copy()); ep_rew = 0.0
        for _ in range(max_len):
            st = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            a  = actor(st).cpu().numpy()
            states, rws, done, _ = env.step(a)
            ep_rew += float(rws.mean())
            if done: break
        rewards.append(ep_rew)
    return float(np.mean(rewards))


# ── 15.3 Evolutionary CMA-ES-like search ─────────────────────────────────────
def evo_tune_action_bias(base_actor, env, init_states, cfg,
                         pop_size=16, elite_frac=0.3, n_iters=60,
                         bias_std_init=0.04, bias_std_min=0.005,
                         bias_std_decay=0.97,
                         eval_episodes_fast=3, eval_episodes_refine=6):
    """
    Population-based search in the ACTION_DIM-dimensional bias space.
    Combines fast evaluation of all candidates + refined evaluation of elites.
    Applies CMA-ES-style mean/sigma update.
    """
    A       = env.adim
    mu      = np.zeros(A,  dtype=np.float32)
    sigma   = np.ones(A,   dtype=np.float32) * bias_std_init
    n_elite = max(1, int(pop_size * elite_frac))
    records = []
    global_best_bias  = mu.copy()
    global_best_score = -np.inf

    for it in range(n_iters):
        cands, scores = [], []

        # Fast evaluation of all candidates
        for _ in range(pop_size):
            eps   = np.random.randn(A).astype(np.float32) * sigma
            bias  = mu + eps
            score = evaluate_bias_candidate(
                bias, base_actor, env, init_states, eval_episodes_fast)
            cands.append(bias); scores.append(score)

        scores_np  = np.array(scores, dtype=np.float32)
        elite_idx  = np.argsort(scores_np)[-(n_elite):]

        # Refined evaluation of elites
        for i in elite_idx:
            score_ref  = evaluate_bias_candidate(
                cands[i], base_actor, env, init_states, eval_episodes_refine)
            scores_np[i] = (scores_np[i] + score_ref) / 2.0

        # Track global best
        it_best_idx   = int(np.argmax(scores_np))
        it_best_score = float(scores_np[it_best_idx])
        if it_best_score > global_best_score:
            global_best_score = it_best_score
            global_best_bias  = cands[it_best_idx].copy()

        # CMA-ES update
        elites = np.stack([cands[i] for i in elite_idx], axis=0)
        mu     = elites.mean(axis=0)
        elite_std = elites.std(axis=0)
        sigma  = (sigma * bias_std_decay +
                  elite_std * (1.0 - bias_std_decay)).clip(bias_std_min, None)

        records.append({"iter": it, "score_mean": float(scores_np.mean()),
                         "score_std":  float(scores_np.std()),
                         "score_best": float(scores_np.max()),
                         "sigma_mean": float(sigma.mean())})
        if (it+1) % 5 == 0:
            print(f"  Evo-Bias it{it+1:02d}  "
                  f"reward_mean={scores_np.mean():.4f}  "
                  f"reward_best={scores_np.max():.4f}  "
                  f"σ={sigma.mean():.4f}")

    best_bias = global_best_bias.astype(np.float32)
    evo_actor = BiasWrapper(base_actor, env.adim)
    with torch.no_grad():
        evo_actor.bias.data.copy_(
            torch.tensor(best_bias, dtype=torch.float32, device=DEVICE))
    evo_actor.to(DEVICE)
    evo_log = pd.DataFrame(records)
    return evo_actor, evo_log


# ── 15.4 Run Evo-MATD3-SR ─────────────────────────────────────────────────────
# Reload best MATD3 actor as frozen base
_base_actor = SharedActor(env_train.sdim, env_train.adim, CFG.MATD3_HIDDEN).to(DEVICE)
_base_actor.load_state_dict(
    torch.load("output/checkpoints/matd3_actor_best.pt", map_location=DEVICE))
for p in _base_actor.parameters():
    p.requires_grad_(False)

with Timer("Evo-MATD3-SR"):
    evo_actor, evo_bias_log = evo_tune_action_bias(
        base_actor          = _base_actor,
        env                 = env_train,
        init_states         = init_tr,
        cfg                 = CFG,
        pop_size            = CFG.EVO_POP_SIZE,
        elite_frac          = CFG.EVO_ELITE_FRAC,
        n_iters             = CFG.EVO_NITERS,
        bias_std_init       = CFG.EVO_BIAS_STD_INIT,
        bias_std_min        = CFG.EVO_BIAS_STD_MIN,
        bias_std_decay      = CFG.EVO_BIAS_STD_DECAY,
        eval_episodes_fast  = CFG.EVO_EVAL_FAST,
        eval_episodes_refine= CFG.EVO_EVAL_REFINE,
    )
    torch.save(evo_actor.state_dict(), "output/checkpoints/evo_matd3sr_bias_actor.pt")
    AM.save_table(evo_bias_log, "05_evo_bias_log")
    print(f"  Evo-MATD3-SR bias: "
          f"mean={evo_actor.bias.data.mean():.4f}  "
          f"std={evo_actor.bias.data.std():.4f}")

# Evo convergence plot
fig, ax = plt.subplots(figsize=(8,4))
ax.fill_between(evo_bias_log["iter"],
                evo_bias_log["score_mean"] - evo_bias_log["score_std"],
                evo_bias_log["score_mean"] + evo_bias_log["score_std"],
                alpha=0.2, color="green")
ax.plot(evo_bias_log["iter"], evo_bias_log["score_mean"],
        color="green", lw=2, label="Mean reward")
ax.plot(evo_bias_log["iter"], evo_bias_log["score_best"],
        color="darkgreen", lw=1.5, ls="--", label="Best reward")
ax.set_xlabel("Evo iteration"); ax.set_ylabel("Train reward")
ax.set_title("Evo-MATD3-SR: Action-Bias Convergence"); ax.legend()
ax.grid(True, alpha=0.4)
AM.save_figure(fig, "06_evo_convergence")
print("✓ Evo-MATD3-SR done")


  Evo-Bias it05  reward_mean=0.2351  reward_best=0.2366  σ=0.0594
  Evo-Bias it10  reward_mean=0.2383  reward_best=0.2397  σ=0.0590
  Evo-Bias it15  reward_mean=0.2412  reward_best=0.2427  σ=0.0586
  Evo-Bias it20  reward_mean=0.2437  reward_best=0.2448  σ=0.0580
  Evo-Bias it25  reward_mean=0.2468  reward_best=0.2484  σ=0.0574
  Evo-Bias it30  reward_mean=0.2493  reward_best=0.2502  σ=0.0570
  Evo-Bias it35  reward_mean=0.2512  reward_best=0.2523  σ=0.0568
  Evo-Bias it40  reward_mean=0.2533  reward_best=0.2546  σ=0.0561
  Evo-Bias it45  reward_mean=0.2550  reward_best=0.2562  σ=0.0554
  Evo-Bias it50  reward_mean=0.2567  reward_best=0.2579  σ=0.0551
  Evo-Bias it55  reward_mean=0.2581  reward_best=0.2591  σ=0.0547
  Evo-Bias it60  reward_mean=0.2595  reward_best=0.2602  σ=0.0543
  Evo-Bias it65  reward_mean=0.2605  reward_best=0.2611  σ=0.0538
  Evo-Bias it70  reward_mean=0.2610  reward_best=0.2616  σ=0.0532
  Evo-Bias it75  reward_mean=0.2615  reward_best=0.2621  σ=0.0529
  Evo-Bias

## Cell 16 · Algorithm Comparison (test set, 10 episodes)

In [16]:
def evaluate_agent(actor, env, init_states, n_episodes=10, label="Agent"):
    """Roll out n_episodes on test env; return mean/std reward & mean actions."""
    all_rewards, all_actions = [], []
    for ep in range(n_episodes):
        states = env.reset(init_states.copy()); ep_rew = 0.0; ep_acts = []
        for _ in range(env.ep_len):
            st   = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts = actor(st).cpu().numpy()
            states, rws, done, _ = env.step(acts)
            ep_rew += float(rws.mean()); ep_acts.append(acts.mean(axis=0))
            if done: break
        all_rewards.append(ep_rew)
        all_actions.extend(ep_acts)
    return {"label": label,
            "mean_rew": float(np.mean(all_rewards)),
            "std_rew":  float(np.std(all_rewards)),
            "mean_act": np.mean(all_actions, axis=0)
                        if all_actions else np.zeros(CFG.ACTION_DIM)}

# Zero-action baseline
class ZeroActor(nn.Module):
    def __init__(self, sdim, adim):
        super().__init__()
        self._a = adim
    def forward(self, s):
        return torch.zeros(s.shape[0], self._a, device=s.device)

zero_actor = ZeroActor(CFG.STATE_DIM, CFG.ACTION_DIM).to(DEVICE)

print("Evaluating on test set (10 episodes each)...")
results = []
for actor_obj, label in [
    (zero_actor,    "Baseline (no action)"),
    (maddpg_actor,  "MADDPG"),
    (matd3_actor,   "MATD3"),
    (evo_actor,     "Evo-MATD3-SR"),
]:
    r = evaluate_agent(actor_obj, env_test, init_te, n_episodes=10, label=label)
    results.append(r)
    print(f"  {label:30s}  reward={r['mean_rew']:.4f} ± {r['std_rew']:.4f}")

res_df = pd.DataFrame([{
    "Algorithm": r["label"],
    "Mean Reward": round(r["mean_rew"], 4),
    "Std Reward":  round(r["std_rew"],  4),
} for r in results])
AM.save_table(res_df, "06_algorithm_comparison")
AM.show_table(res_df, "Algorithm Comparison (test set)")

# Bar chart
labels = [r["label"] for r in results]
means  = [r["mean_rew"] for r in results]
stds   = [r["std_rew"]  for r in results]
colors = ["#aaaaaa","#4c72b0","#dd8452","#55a868"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors,
              alpha=0.85, edgecolor="black")
ax.set_ylabel("Mean Episodic Reward"); ax.set_title("Algorithm Comparison – test set (10 eps)")
ax.tick_params(axis="x", rotation=15)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.002,
            f"{m:.4f}", ha="center", va="bottom", fontsize=9)
ax.grid(True, alpha=0.4, axis="y")
plt.tight_layout()
AM.save_figure(fig, "07_algorithm_comparison")
print("  → output/figures/07_algorithm_comparison.png")
print("✓ Algorithm comparison done")


Evaluating on test set (10 episodes each)...
  Baseline (no action)            reward=0.2644 ± 0.0016
  MADDPG                          reward=0.2766 ± 0.0020
  MATD3                           reward=0.2670 ± 0.0012
  Evo-MATD3-SR                    reward=0.3136 ± 0.0010


,Algorithm,Mean Reward,Std Reward
0,Baseline (no action),0.2644,0.0016
1,MADDPG,0.2766,0.0020
2,MATD3,0.2670,0.0012
3,Evo-MATD3-SR,0.3136,0.0010


  → output/figures/07_algorithm_comparison.png
✓ Algorithm comparison done


## Cell 17 · Action Profile Analysis: what policies do agents choose?

In [17]:
# Radar / bar chart of mean actions per algorithm
act_data = {r["label"]: r["mean_act"] for r in results if r["label"] != "Baseline (no action)"}
action_names_short = [n.replace("_"," ").title() for n in CFG.ACTION_NAMES]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Mean Action Profiles by Algorithm", fontsize=12, fontweight="bold")

for ax, (alg, acts) in zip(axes, act_data.items()):
    bars = ax.bar(range(len(acts)), acts,
                  color=sns.color_palette("viridis", len(acts)), alpha=0.85)
    ax.set_xticks(range(len(acts)))
    ax.set_xticklabels([f"A{i+1:02d}" for i in range(len(acts))],
                       rotation=45, fontsize=7)
    ax.set_ylim(0, 0.5)
    ax.set_title(alg, fontsize=9)
    ax.set_ylabel("Mean action value"); ax.grid(True, alpha=0.3, axis="y")
    # Annotate top actions
    top3 = np.argsort(acts)[-3:][::-1]
    for idx in top3:
        ax.text(idx, acts[idx]+0.01, action_names_short[idx],
                ha="center", fontsize=6, color="darkred", rotation=30)

plt.tight_layout()
AM.save_figure(fig, "08_action_profiles")
print("  → output/figures/08_action_profiles.png")

# Table: action means per algorithm
act_df = pd.DataFrame({alg: acts for alg, acts in act_data.items()},
                      index=action_names_short)
AM.save_table(act_df, "07_action_profiles")
AM.show_table(act_df.round(4), "Action profiles (mean per algorithm)")
print("✓ Action profile analysis done")


  → output/figures/08_action_profiles.png


,MADDPG,MATD3,Evo-MATD3-SR
Trade Openness,0.7120,0.4975,1.0000
Military Buildup,0.1444,0.2004,0.0014
Rnd Investment,0.3683,0.6693,0.0018
Digitalisation,0.5653,0.7000,0.0000
Education Policy,0.5783,0.5371,0.2100
Social Equity,0.2320,0.3966,0.0000
Climate Energy,0.2687,0.6324,1.0000
Infrastructure,0.3326,0.7383,0.0000
Fiscal Stimulus,0.3892,0.6382,0.1767
Fdi Attraction,0.5894,0.6435,0.0000


✓ Action profile analysis done


## Cell 18 · Extended Crisis Palette (30 scenarios, past & future) + Stress Test

In [18]:
# ══════════════════════════════════════════════════════════════════════════
# 30 CRISIS SCENARIOS — past (empirically grounded) + future (plausible)
# 4-vector: (gdp_shock, mortality_shock, migration_shock, energy_shock)
# All values are NORMALISED deviations from baseline (relative, not absolute).
# References: UCDP, World Bank GFC data, IPCC AR6, WEF Global Risks Report.
# ══════════════════════════════════════════════════════════════════════════
FULL_CRISIS_PALETTE = [
    # ── PAST CRISES (historical, empirically grounded) ───────────────────
    {"id":  1, "era":"past",   "name":"Asian Financial Crisis 1997-98",
     "b":-0.07,"d":0.04,"m":-0.20,"e":-0.08},
    {"id":  2, "era":"past",   "name":"Dot-com Bust & 9/11 Shock 2001",
     "b":-0.06,"d":0.03,"m":-0.12,"e":-0.05},
    {"id":  3, "era":"past",   "name":"SARS Pandemic 2003",
     "b":-0.04,"d":0.06,"m":-0.15,"e":-0.03},
    {"id":  4, "era":"past",   "name":"Global Financial Crisis 2008-09",
     "b":-0.25,"d":0.08,"m":-0.40,"e":-0.30},
    {"id":  5, "era":"past",   "name":"Eurozone Sovereign Debt Crisis 2011-12",
     "b":-0.12,"d":0.05,"m":-0.25,"e":-0.15},
    {"id":  6, "era":"past",   "name":"Commodity Price Collapse 2014-16",
     "b":-0.10,"d":0.03,"m":-0.20,"e":-0.35},
    {"id":  7, "era":"past",   "name":"COVID-19 Pandemic 2020",
     "b":-0.18,"d":0.28,"m":-0.65,"e":-0.22},
    {"id":  8, "era":"past",   "name":"Supply Chain Disruption 2021",
     "b":-0.05,"d":0.08,"m":-0.25,"e":-0.18},
    {"id":  9, "era":"past",   "name":"Global Inflation & Energy Shock 2022",
     "b":-0.08,"d":0.04,"m":-0.20,"e":-0.55},
    {"id": 10, "era":"past",   "name":"Sectoral Sanctions Regime 2022-24",
     "b":-0.14,"d":0.03,"m":-0.35,"e":-0.40},
    {"id": 11, "era":"past",   "name":"Arab Spring & Regional Instability 2011",
     "b":-0.09,"d":0.12,"m":-0.50,"e":-0.10},
    {"id": 12, "era":"past",   "name":"Ebola Outbreak (West Africa) 2014-16",
     "b":-0.05,"d":0.18,"m":-0.30,"e":-0.05},
    {"id": 13, "era":"past",   "name":"Latin America Debt Crisis 1982",
     "b":-0.15,"d":0.06,"m":-0.22,"e":-0.10},
    {"id": 14, "era":"past",   "name":"Post-Soviet Transition Shock 1991-95",
     "b":-0.20,"d":0.12,"m":-0.45,"e":-0.25},
    {"id": 15, "era":"past",   "name":"Food Price Spike 2007-08",
     "b":-0.04,"d":0.09,"m":-0.18,"e":-0.06},
    # ── FUTURE SCENARIOS (plausible, IPCC/WEF/RAND grounded) ────────────
    {"id": 16, "era":"future", "name":"Pandemic X (Novel Pathogen 2030s)",
     "b":-0.20,"d":0.35,"m":-0.70,"e":-0.25},
    {"id": 17, "era":"future", "name":"Climate Tipping Point – Extreme Heat 2035",
     "b":-0.15,"d":0.20,"m":-0.55,"e":-0.30},
    {"id": 18, "era":"future", "name":"Global Cyberattack on Critical Infrastructure",
     "b":-0.12,"d":0.10,"m":-0.20,"e":-0.50},
    {"id": 19, "era":"future", "name":"AI-Driven Labour Market Disruption 2030",
     "b":-0.10,"d":0.02,"m":-0.15,"e":-0.05},
    {"id": 20, "era":"future", "name":"Deglobalisation & Trade Bloc Fragmentation",
     "b":-0.18,"d":0.03,"m":-0.30,"e":-0.35},
    {"id": 21, "era":"future", "name":"Fresh Water Scarcity Shock 2040",
     "b":-0.08,"d":0.15,"m":-0.40,"e":-0.10},
    {"id": 22, "era":"future", "name":"Semiconductor Supply Collapse 2028",
     "b":-0.14,"d":0.02,"m":-0.10,"e":-0.20},
    {"id": 23, "era":"future", "name":"Sovereign Debt Cascade (EM economies) 2032",
     "b":-0.22,"d":0.07,"m":-0.35,"e":-0.25},
    {"id": 24, "era":"future", "name":"Arctic Resource Rush & Geopolitical Conflict 2038",
     "b":-0.09,"d":0.08,"m":-0.20,"e":-0.45},
    {"id": 25, "era":"future", "name":"Global Food System Shock (Crop Failure) 2034",
     "b":-0.11,"d":0.22,"m":-0.45,"e":-0.12},
    {"id": 26, "era":"future", "name":"Nuclear Proliferation Regional Escalation 2036",
     "b":-0.25,"d":0.20,"m":-0.60,"e":-0.50},
    {"id": 27, "era":"future", "name":"Rapid De-dollarisation & Currency Shock 2030",
     "b":-0.13,"d":0.02,"m":-0.15,"e":-0.20},
    {"id": 28, "era":"future", "name":"Antibiotic Resistance Health Crisis 2040",
     "b":-0.06,"d":0.25,"m":-0.20,"e":-0.04},
    {"id": 29, "era":"future", "name":"Loss of Biodiversity & Ecosystem Services 2045",
     "b":-0.07,"d":0.12,"m":-0.25,"e":-0.08},
    {"id": 30, "era":"future", "name":"Space-Weather / Solar Flare Grid Collapse 2042",
     "b":-0.10,"d":0.06,"m":-0.12,"e":-0.60},
]
print(f"Crisis palette loaded: {len(FULL_CRISIS_PALETTE)} scenarios "
      f"({sum(1 for c in FULL_CRISIS_PALETTE if c['era']=='past')} past + "
      f"{sum(1 for c in FULL_CRISIS_PALETTE if c['era']=='future')} future)")

# ── Stress-test function ──────────────────────────────────────────────────────
def crisis_stress_test(actor, env, init_states, crisis_list, n_reps=3):
    """Evaluate each crisis scenario: mean reward over n_reps episodes."""
    records = []
    for crisis in crisis_list:
        ep_rewards = []
        for _ in range(n_reps):
            states = env.reset(init_states.copy()); ep_rew = 0.0
            # Inject crisis for entire episode
            env.crisis   = crisis
            env.crisis_t = env.ep_len + 1
            for _ in range(env.ep_len):
                st   = torch.tensor(states, dtype=torch.float32, device=DEVICE)
                actor.eval()
                with torch.no_grad():
                    acts = actor(st).cpu().numpy()
                states, rws, done, _ = env.step(acts)
                ep_rew += float(rws.mean())
                if done: break
            ep_rewards.append(ep_rew)
        records.append({
            "crisis_id":   crisis["id"],
            "crisis_name": crisis["name"],
            "era":         crisis["era"],
            "mean_rew":    float(np.mean(ep_rewards)),
            "std_rew":     float(np.std(ep_rewards)),
        })
    return pd.DataFrame(records)

print("Running stress tests for all 30 crises × 3 algorithms ...")
stress_dfs = []
for actor_obj, alg_label in [
    (maddpg_actor, "MADDPG"),
    (matd3_actor,  "MATD3"),
    (evo_actor,    "Evo-MATD3-SR"),
]:
    df_s = crisis_stress_test(actor_obj, env_test, init_te,
                               FULL_CRISIS_PALETTE, n_reps=3)
    df_s["algorithm"] = alg_label
    stress_dfs.append(df_s)
    print(f"  {alg_label}: done")

df_stress_all = pd.concat(stress_dfs, ignore_index=True)
AM.save_table(df_stress_all, "08_crisis_stress_test")

# ── Heatmap ───────────────────────────────────────────────────────────────────
stress_pivot = df_stress_all.pivot(index="crisis_name", columns="algorithm",
                                   values="mean_rew")
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(stress_pivot, annot=True, fmt=".2f", cmap="RdYlGn",
            linewidths=0.4, ax=ax, cbar_kws={"label":"Mean Reward"})
ax.set_title("Crisis Stress Test — Mean Reward (30 scenarios × 3 algorithms)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Algorithm"); ax.set_ylabel("Crisis Scenario")
plt.tight_layout()
AM.save_figure(fig, "09_crisis_stress_heatmap")
print("  → output/figures/09_crisis_stress_heatmap.png")

# ── Grouped bar: past vs future ───────────────────────────────────────────────
era_summary = (df_stress_all.groupby(["era","algorithm"])["mean_rew"]
               .mean().unstack("algorithm"))
fig2, ax2 = plt.subplots(figsize=(8,5))
era_summary.plot(kind="bar", ax=ax2, colormap="Set2", edgecolor="black",
                 alpha=0.85, rot=0)
ax2.set_title("Mean Reward: Past vs Future Crisis Scenarios")
ax2.set_ylabel("Mean Reward"); ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.4, axis="y")
plt.tight_layout()
AM.save_figure(fig2, "09b_crisis_past_vs_future")
AM.show_table(stress_pivot.round(3), "Crisis Stress Test Heatmap")
print("✓ Extended crisis stress test done")


Crisis palette loaded: 30 scenarios (15 past + 15 future)
Running stress tests for all 30 crises × 3 algorithms ...
  MADDPG: done
  MATD3: done
  Evo-MATD3-SR: done
  → output/figures/09_crisis_stress_heatmap.png


algorithm,Evo-MATD3-SR,MADDPG,MATD3
crisis_name,,,
AI-Driven Labour Market Disruption 2030,0.3130,0.2760,0.2670
Antibiotic Resistance Health Crisis 2040,0.3120,0.2760,0.2660
Arab Spring & Regional Instability 2011,0.3130,0.2760,0.2660
Arctic Resource Rush & Geopolitical Conflict 2038,0.3110,0.2750,0.2660
Asian Financial Crisis 1997-98,0.3140,0.2760,0.2670
COVID-19 Pandemic 2020,0.3090,0.2730,0.2640
Climate Tipping Point – Extreme Heat 2035,0.3090,0.2740,0.2650
Commodity Price Collapse 2014-16,0.3120,0.2740,0.2660
Deglobalisation & Trade Bloc Fragmentation,0.3110,0.2740,0.2650


✓ Extended crisis stress test done


## Cell 19 · Country-Level Analysis: RUS, G7, BRICS+

In [19]:
# ── Country groups ────────────────────────────────────────────────────────────
COUNTRY_GROUPS = {
    "G7":       ["CAN","FRA","DEU","ITA","JPN","GBR","USA"],
    "BRICS+":   ["BRA","RUS","IND","CHN","ZAF","EGY","ETH","IRN","SAU","ARE"],
    "EU_CORE":  ["DEU","FRA","ITA","ESP","NLD","POL","SWE"],
    "ASEAN+":   ["IDN","MYS","THA","PHL","VNM","SGP","KOR"],
    "Spotlight":["RUS","USA","CHN","DEU","BRA","IND","ZAF","TUR","NGA","MEX"],
}

def country_action_profile(actor, env, init_states, country_list, n_eps=5):
    """Return per-country mean actions over n_eps episodes."""
    ctry_acts = {c: [] for c in country_list}
    ctry_rews = {c: [] for c in country_list}
    for ep in range(n_eps):
        states = env.reset(init_states.copy())
        for _ in range(env.ep_len):
            st   = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts = actor(st).cpu().numpy()
            states, rws, done, _ = env.step(acts)
            for i, c in enumerate(env.countries):
                if c in ctry_acts:
                    ctry_acts[c].append(acts[i])
                    ctry_rews[c].append(float(rws[i]))
            if done: break
    return ({c: np.mean(v, axis=0) for c, v in ctry_acts.items() if v},
            {c: float(np.mean(v))  for c, v in ctry_rews.items() if v})

# Build spotlight country list (only those present in model)
spotlight = [c for c in COUNTRY_GROUPS["Spotlight"] if c in countries]
print(f"  Spotlight countries in model: {spotlight}")

print("Computing per-country action profiles (Evo-MATD3-SR) ...")
ctry_acts_evo, ctry_rews_evo = country_action_profile(
    evo_actor, env_test, init_te, spotlight, n_eps=5)

# DataFrame: countries × actions
act_country_df = pd.DataFrame(
    {c: ctry_acts_evo.get(c, np.zeros(CFG.ACTION_DIM)) for c in spotlight},
    index=CFG.ACTION_NAMES).T.round(4)
AM.save_table(act_country_df, "09_country_action_profiles")
AM.show_table(act_country_df, "Per-country action profiles (Evo-MATD3-SR)")

# ── Heatmap: countries × actions ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(act_country_df, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.3, ax=ax, cbar_kws={"label":"Mean action value"})
ax.set_title("Country Action Profiles — Evo-MATD3-SR", fontsize=11)
ax.set_xlabel("Policy instrument"); ax.set_ylabel("Country")
plt.tight_layout()
AM.save_figure(fig, "10_country_action_heatmap")
print("  → output/figures/10_country_action_heatmap.png")

# ── Reward comparison: G7 vs BRICS+ ──────────────────────────────────────────
g7_present    = [c for c in COUNTRY_GROUPS["G7"]    if c in countries]
brics_present = [c for c in COUNTRY_GROUPS["BRICS+"] if c in countries]

_, rews_all_evo = country_action_profile(
    evo_actor, env_test, init_te, g7_present + brics_present, n_eps=5)

g7_rews    = [rews_all_evo.get(c, 0) for c in g7_present]
brics_rews = [rews_all_evo.get(c, 0) for c in brics_present]

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle("Country Rewards — G7 vs BRICS+ (Evo-MATD3-SR)", fontsize=11)
for ax2, grp, rews, label in [
    (axes2[0], g7_present,    g7_rews,    "G7"),
    (axes2[1], brics_present, brics_rews, "BRICS+"),
]:
    colors_g = sns.color_palette("tab10", len(grp))
    ax2.barh(grp, rews, color=colors_g, alpha=0.85, edgecolor="black")
    ax2.set_title(f"{label} — Mean Reward")
    ax2.set_xlabel("Mean Reward")
    ax2.axvline(0, color="gray", lw=1, ls="--")
    ax2.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
AM.save_figure(fig2, "11_g7_vs_brics_rewards")
print("  → output/figures/11_g7_vs_brics_rewards.png")

rew_df = pd.DataFrame([
    {"country": c, "group": "G7",    "mean_reward": rews_all_evo.get(c,0)} for c in g7_present
] + [
    {"country": c, "group": "BRICS+","mean_reward": rews_all_evo.get(c,0)} for c in brics_present
])
AM.save_table(rew_df, "10_g7_brics_rewards")
AM.show_table(rew_df.round(4), "G7 vs BRICS+ rewards (Evo-MATD3-SR)")
print("✓ Country-level analysis done")


  Spotlight countries in model: ['RUS', 'USA', 'CHN', 'DEU', 'BRA', 'IND', 'ZAF', 'TUR', 'NGA', 'MEX']
Computing per-country action profiles (Evo-MATD3-SR) ...


,trade_openness,military_buildup,rnd_investment,digitalisation,education_policy,social_equity,climate_energy,infrastructure,fiscal_stimulus,fdi_attraction,institution_reform,stability_security,alliance_cooperation,sanctions_pressure
RUS,1.0000,0.0000,0.0000,0.0000,0.3915,0.0000,1.0000,0.0000,0.3637,0.0000,0.0000,0.5808,0.2456,1.0000
USA,1.0000,0.0000,0.0039,0.0000,0.4947,0.0000,1.0000,0.0000,0.3695,0.0000,0.0000,0.5835,0.2697,1.0000
CHN,1.0000,0.0000,0.0000,0.0000,0.4256,0.0000,1.0000,0.0000,0.3625,0.0000,0.0000,0.5764,0.1596,1.0000
DEU,1.0000,0.0000,0.0000,0.0000,0.5032,0.0000,1.0000,0.0000,0.1576,0.0000,0.0000,0.5839,0.2715,1.0000
BRA,1.0000,0.0000,0.0000,0.0000,0.4521,0.0000,1.0000,0.0000,0.3724,0.0000,0.0000,0.5818,0.2755,1.0000
IND,1.0000,0.0000,0.0000,0.0000,0.4913,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.5721,0.0568,1.0000
ZAF,1.0000,0.0000,0.0000,0.0000,0.3353,0.0000,1.0000,0.0000,0.1930,0.0000,0.0000,0.5490,0.0762,1.0000
TUR,1.0000,0.0000,0.0108,0.0000,0.2064,0.0000,1.0000,0.0000,0.2687,0.0000,0.0000,0.5725,0.2405,1.0000
NGA,1.0000,0.0000,0.0046,0.0000,0.3297,0.0000,1.0000,0.0000,0.2769,0.0000,0.0000,0.5700,0.2439,1.0000
MEX,1.0000,0.0288,0.0000,0.0000,0.4245,0.0000,1.0000,0.0000,0.3668,0.0000,0.0000,0.5503,0.1511,1.0000


  → output/figures/10_country_action_heatmap.png
  → output/figures/11_g7_vs_brics_rewards.png


,country,group,mean_reward
0,CAN,G7,0.0387
1,FRA,G7,0.0417
2,DEU,G7,0.0542
3,ITA,G7,0.0433
4,JPN,G7,0.0378
5,GBR,G7,0.0536
6,USA,G7,0.0375
7,BRA,BRICS+,0.0287
8,RUS,BRICS+,0.0033
9,IND,BRICS+,-0.0245


✓ Country-level analysis done


## Cell 20 · Russia Spotlight & Political Hypothesis Testing

> **Note on scientific framing.** The analysis below applies the same
> modelling framework uniformly to all countries, including Russia (RUS).
> Governance indicators are sourced from the publicly available WGI database
> (World Bank, 2025) and are treated as statistical estimates, not evaluative
> judgements. Crisis scenarios are labelled generically (e.g. 'Sanctions
> Regime', 'Energy Shock') and are calibrated from IMF/WB historical data.
> All interpretations remain within the scientific and analytical framework
> of computational political science.


In [20]:
# ── Russia-specific crisis set ────────────────────────────────────────────────
RUS_SPECIFIC_CRISES = [
    s for s in FULL_CRISIS_PALETTE if s["name"] in [
        "Sectoral Sanctions Regime 2022-24",
        "Global Inflation & Energy Shock 2022",
        "Energy Price Shock",
        "Commodity Price Collapse 2014-16",
        "Deglobalisation & Trade Bloc Fragmentation",
        "Rapid De-dollarisation & Currency Shock 2030",
        "Post-Soviet Transition Shock 1991-95",
    ]
]
print(f"  Russia-relevant crises: {len(RUS_SPECIFIC_CRISES)}")

# ── Russia action profile across algorithms ───────────────────────────────────
RUS_CODE = "RUS"
rus_acts_by_alg = {}

if RUS_CODE in countries:
    for actor_obj, alg in [(maddpg_actor,"MADDPG"),
                           (matd3_actor,"MATD3"),
                           (evo_actor,"Evo-MATD3-SR")]:
        acts_, rews_ = country_action_profile(
            actor_obj, env_test, init_te, [RUS_CODE], n_eps=5)
        rus_acts_by_alg[alg] = acts_.get(RUS_CODE, np.zeros(CFG.ACTION_DIM))

    rus_act_df = pd.DataFrame(rus_acts_by_alg, index=CFG.ACTION_NAMES)
    AM.save_table(rus_act_df, "11_russia_action_profiles")

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(CFG.ACTION_DIM); w = 0.25
    for ki, (alg, vals) in enumerate(rus_acts_by_alg.items()):
        ax.bar(x + ki*w, vals, width=w, label=alg, alpha=0.8)
    ax.set_xticks(x + w)
    ax.set_xticklabels([f"A{i+1:02d}" for i in range(CFG.ACTION_DIM)],
                       rotation=45, fontsize=7)
    ax.set_ylabel("Mean action value")
    ax.set_title("Russia (RUS) — Policy Instrument Profiles by Algorithm")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")
    # Annotate action names
    for i, aname in enumerate(CFG.ACTION_NAMES):
        ax.text(i+w, -0.04, aname.replace("_","\n"), ha="center",
                fontsize=5.5, rotation=60, transform=ax.get_xaxis_transform())
    plt.tight_layout()
    AM.save_figure(fig, "12_russia_action_profiles")
    print("  → output/figures/12_russia_action_profiles.png")
    AM.show_table(rus_act_df.round(4), "Russia Policy Profiles by Algorithm")
else:
    print(f"  ⚠️  {RUS_CODE} not in model countries – check data coverage")

# ── Russia stress: crises vs algorithms ───────────────────────────────────────
if RUS_CODE in countries and RUS_SPECIFIC_CRISES:
    def country_crisis_reward(actor, env, init_states, country_code,
                              crisis_list, n_reps=3):
        """Reward of one specific country under each crisis."""
        ci = env.countries.index(country_code) if country_code in env.countries else -1
        if ci < 0:
            return pd.DataFrame()
        records = []
        for crisis in crisis_list:
            rws_list = []
            for _ in range(n_reps):
                states = env.reset(init_states.copy()); ep_rew = 0.0
                env.crisis = crisis; env.crisis_t = env.ep_len + 1
                for _ in range(env.ep_len):
                    st = torch.tensor(states, dtype=torch.float32, device=DEVICE)
                    actor.eval()
                    with torch.no_grad():
                        acts = actor(st).cpu().numpy()
                    states, rws, done, _ = env.step(acts)
                    ep_rew += float(rws[ci])
                    if done: break
                rws_list.append(ep_rew)
            records.append({"crisis": crisis["name"],
                            "mean_rew": float(np.mean(rws_list)),
                            "std_rew":  float(np.std(rws_list))})
        return pd.DataFrame(records)

    rus_stress = {}
    for actor_obj, alg in [(maddpg_actor,"MADDPG"),
                           (matd3_actor,"MATD3"),
                           (evo_actor,"Evo-MATD3-SR")]:
        df_rs = country_crisis_reward(actor_obj, env_test, init_te,
                                      RUS_CODE, RUS_SPECIFIC_CRISES, n_reps=3)
        df_rs["algorithm"] = alg
        rus_stress[alg]    = df_rs

    df_rus_stress = pd.concat(rus_stress.values(), ignore_index=True)
    AM.save_table(df_rus_stress, "12_russia_crisis_stress")

    rus_pivot = df_rus_stress.pivot(index="crisis", columns="algorithm",
                                    values="mean_rew")
    fig2, ax2 = plt.subplots(figsize=(10,5))
    sns.heatmap(rus_pivot, annot=True, fmt=".2f", cmap="RdYlGn",
                linewidths=0.4, ax=ax2)
    ax2.set_title("Russia — Mean Reward per Crisis Scenario (3 algorithms)",
                  fontsize=10, fontweight="bold")
    plt.tight_layout()
    AM.save_figure(fig2, "13_russia_crisis_heatmap")
    print("  → output/figures/13_russia_crisis_heatmap.png")
    AM.show_table(rus_pivot.round(3), "Russia Crisis Stress (pivot)")
print("✓ Russia spotlight done")


  Russia-relevant crises: 6
  → output/figures/12_russia_action_profiles.png


,MADDPG,MATD3,Evo-MATD3-SR
trade_openness,0.2622,0.6738,1.0000
military_buildup,0.0627,0.2110,0.0000
rnd_investment,0.8742,0.8179,0.0000
digitalisation,0.7273,0.7596,0.0000
education_policy,0.0441,0.8806,0.3911
social_equity,0.7769,0.2119,0.0000
climate_energy,0.8349,0.7463,1.0000
infrastructure,0.8885,0.2941,0.0000
fiscal_stimulus,0.4838,0.9785,0.3633
fdi_attraction,0.7747,0.9968,0.0000


  → output/figures/13_russia_crisis_heatmap.png


algorithm,Evo-MATD3-SR,MADDPG,MATD3
crisis,,,
Commodity Price Collapse 2014-16,0.0580,0.0150,0.0210
Deglobalisation & Trade Bloc Fragmentation,0.0570,0.0170,0.0240
Global Inflation & Energy Shock 2022,0.0660,0.0150,0.0300
Post-Soviet Transition Shock 1991-95,0.0610,0.0160,0.0260
Rapid De-dollarisation & Currency Shock 2030,0.0570,0.0190,0.0300
Sectoral Sanctions Regime 2022-24,0.0620,0.0110,0.0310


✓ Russia spotlight done


## Cell 21 · Hypothesis Testing: 12 Political-Science Hypotheses

In [21]:
# ── Political-science hypotheses (12 + 2 Russia-specific) ────────────────────
HYPOTHESES = {
    "H1":  "Countries with higher governance quality choose more cooperative strategies"
           " (higher A01/A13, lower A14).",
    "H2":  "Political stability (pv_score) buffers against crisis shocks "
           "(lower reward variance under crises).",
    "H3":  "Resource-dependent economies (high nat_resources_gdp) require higher "
           "R&D investment (A03) for long-run reward.",
    "H4":  "Digitalisation (A04) substitutes for governance deficit: "
           "countries with low ge_score gain more from A04.",
    "H5":  "High education investment (A05) improves political stability "
           "over multi-step horizon.",
    "H6":  "Military buildup (A02) yields short-term stability gains but "
           "reduces long-run reward.",
    "H7":  "Governance quality amplifies returns to R&D (A03) and "
           "digitalisation (A04).",
    "H8":  "Female labour participation growth (A06) improves long-run "
           "resilience and reward stability.",
    "H9":  "Aggressive sanctions use (high A14) leads to bloc formation "
           "(structural change in adjacency clusters) and lower global reward.",
    "H10": "Evo-MATD3-SR identifies structurally distinct 'world order' "
           "modes vs local MARL optima (MADDPG/MATD3).",
    "H11_RUS": "For Russia, energy policy instruments (A07/A12) carry higher "
               "marginal reward under energy-shock crises than global average.",
    "H12_RUS": "For Russia, institutional reform actions (A11/A12) show "
               "higher marginal reward in scenarios with external economic pressure.",
}

# ── H1: Governance → cooperation ─────────────────────────────────────────────
h1_records = []
if wgi_present:
    gov_mean = df_norm.groupby("country_code")[wgi_present].mean().mean(axis=1)
    for c in countries:
        gv = gov_mean.get(c, 0)
        acts = ctry_acts_evo.get(c, None)
        if acts is not None:
            h1_records.append({
                "country": c,
                "governance_mean": float(gv),
                "A01_trade_open":  float(acts[0]),
                "A13_alliance":    float(acts[12]),
                "A14_sanctions":   float(acts[13]),
                "coop_score":      float(acts[0] + acts[12] - acts[13]),
            })

h1_df = pd.DataFrame(h1_records)
if len(h1_df) > 5:
    r_h1, p_h1 = stats.pearsonr(h1_df["governance_mean"], h1_df["coop_score"])
    print(f"H1 — Governance ↔ Cooperation: r={r_h1:.3f}, p={p_h1:.4f}  "
          f"({'SUPPORTED' if p_h1<0.05 else 'not supported'})")
    AM.save_table(h1_df, "13_H1_governance_cooperation")

# ── H2: pv_score → crisis resilience ─────────────────────────────────────────
if "pv_score" in df_norm.columns:
    pv_mean = df_norm.groupby("country_code")["pv_score"].mean()
    # Compute reward std across all crisis scenarios per country
    crisis_rew_std = (df_stress_all[df_stress_all["algorithm"]=="Evo-MATD3-SR"]
                     .groupby("crisis_name")["std_rew"].mean())
    print(f"H2 — Political Stability crisis resilience: "
          f"mean crisis std_rew={crisis_rew_std.mean():.4f}")

# ── H9: Sanctions → bloc formation ───────────────────────────────────────────
# Proxy: variance of A14 across countries
if ctry_acts_evo:
    a14_vals = [v[13] for v in ctry_acts_evo.values()]
    print(f"H9 — Sanctions use (A14): mean={np.mean(a14_vals):.3f}, "
          f"std={np.std(a14_vals):.3f}  "
          f"({'High dispersion – bloc tendencies' if np.std(a14_vals)>0.08 else 'Low dispersion'})")

# ── H10: Evo vs MADDPG/MATD3 bias diversity ──────────────────────────────────
evo_bias = evo_actor.bias.data.cpu().numpy()
print(f"H10 — Evo bias vector: {np.round(evo_bias,3)}")
print(f"       Top positive actions: "
      f"{[CFG.ACTION_NAMES[i] for i in np.argsort(evo_bias)[-3:][::-1]]}")
print(f"       Top suppressed actions: "
      f"{[CFG.ACTION_NAMES[i] for i in np.argsort(evo_bias)[:3]]}")

# ── Hypothesis summary table ──────────────────────────────────────────────────
hyp_df = pd.DataFrame([
    {"Hypothesis": k, "Statement": v} for k, v in HYPOTHESES.items()
])
AM.save_table(hyp_df, "14_hypothesis_list")
AM.show_table(hyp_df, "Political Science Hypotheses")
print("✓ Hypothesis testing done")


H1 — Governance ↔ Cooperation: r=nan, p=nan  (not supported)
H2 — Political Stability crisis resilience: mean crisis std_rew=0.0005
H9 — Sanctions use (A14): mean=1.000, std=0.000  (Low dispersion)
H10 — Evo bias vector: [ 1.108 -0.815 -0.979 -1.075 -0.489 -1.09   1.151 -1.077 -0.614 -1.198
 -1.062 -0.415 -0.712  1.021]
       Top positive actions: ['climate_energy', 'trade_openness', 'sanctions_pressure']
       Top suppressed actions: ['fdi_attraction', 'social_equity', 'infrastructure']


,Hypothesis,Statement
0,H1,"Countries with higher governance quality choose more cooperative strategies (higher A01/A13, lower A14)."
1,H2,Political stability (pv_score) buffers against crisis shocks (lower reward variance under crises).
2,H3,Resource-dependent economies (high nat_resources_gdp) require higher R&D investment (A03) for long-run reward.
3,H4,Digitalisation (A04) substitutes for governance deficit: countries with low ge_score gain more from A04.
4,H5,High education investment (A05) improves political stability over multi-step horizon.
5,H6,Military buildup (A02) yields short-term stability gains but reduces long-run reward.
6,H7,Governance quality amplifies returns to R&D (A03) and digitalisation (A04).
7,H8,Female labour participation growth (A06) improves long-run resilience and reward stability.
8,H9,Aggressive sanctions use (high A14) leads to bloc formation (structural change in adjacency clusters) and lower global reward.
9,H10,Evo-MATD3-SR identifies structurally distinct 'world order' modes vs local MARL optima (MADDPG/MATD3).


✓ Hypothesis testing done


## Cell 22 · Trajectory Simulation 2025-2050 (all algorithms)

In [22]:
# ── Simulate multi-step trajectories ─────────────────────────────────────────
def simulate_trajectory(actor, env, init_states, sim_years, feat_cols,
                         country_list, policy_label="Policy"):
    """One trajectory: step through sim_years; record per-country, per-feature states."""
    records = []
    states  = env.reset(init_states.copy())
    for yr in sim_years:
        st = torch.tensor(states, dtype=torch.float32, device=DEVICE)
        actor.eval()
        with torch.no_grad():
            acts = actor(st).cpu().numpy()
        next_s, rws, done, info = env.step(acts)
        for ri, ctry in enumerate(country_list):
            row = {"country_code": ctry, "year": yr,
                   "algorithm": policy_label,
                   "reward": float(rws[ri]),
                   "crisis": info.get("crisis")}
            n_feats = min(len(feat_cols), states.shape[1])
            for fi, feat in enumerate(feat_cols[:n_feats]):
                row[feat] = float(states[ri, fi])
            records.append(row)
        states = next_s
    return pd.DataFrame(records)

SIM_YEARS = list(range(2025, 2051))      # 26 years → horizon to 2050
print(f"Simulating trajectories for {len(SIM_YEARS)} years (2025-2050) ...")

traj_parts = []
for actor_obj, alg in [
    (zero_actor,   "Baseline"),
    (maddpg_actor, "MADDPG"),
    (matd3_actor,  "MATD3"),
    (evo_actor,    "Evo-MATD3-SR"),
]:
    with Timer(f"  Traj {alg}"):
        df_t = simulate_trajectory(
            actor_obj, env_test, init_te,
            SIM_YEARS, FEAT_COLS, countries, policy_label=alg)
        traj_parts.append(df_t)

df_traj_all = pd.concat(traj_parts, ignore_index=True)
print(f"  Trajectory dataframe: {df_traj_all.shape}")
AM.save_table(df_traj_all, "15_simulated_trajectories_2025_2050")

# ── Plot key indicators: spotlight countries ──────────────────────────────────
FEATS_PLOT  = [f for f in ["gdp_growth","governance_mean",
                            "internet_users","renewable_energy",
                            "trade_gdp","gini_index"]
               if f in FEAT_COLS][:4]
SHOWCASE2   = [c for c in ["RUS","USA","CHN","DEU","BRA","IND"]
               if c in countries][:4]
ALG_COLORS  = {"Baseline":"gray","MADDPG":"#4c72b0",
               "MATD3":"#dd8452","Evo-MATD3-SR":"#55a868"}
ALG_STYLES  = {"Baseline":"--","MADDPG":"-","MATD3":"-","Evo-MATD3-SR":"-"}
ALG_WIDTH   = {"Baseline":1.0,"MADDPG":1.5,"MATD3":1.5,"Evo-MATD3-SR":2.0}

fig, axes = plt.subplots(len(FEATS_PLOT), len(SHOWCASE2),
                          figsize=(16, 3.5*len(FEATS_PLOT)))
fig.suptitle("Simulated Trajectories 2025-2050: 4 algorithms × 4 countries × 4 indicators",
             fontsize=11, fontweight="bold")

for ai, feat in enumerate(FEATS_PLOT):
    for ci, ctry in enumerate(SHOWCASE2):
        ax = axes[ai][ci] if len(FEATS_PLOT) > 1 else axes[ci]
        for alg in ["Baseline","MADDPG","MATD3","Evo-MATD3-SR"]:
            sub = df_traj_all[(df_traj_all["country_code"]==ctry) &
                              (df_traj_all["algorithm"]==alg)]
            if len(sub) > 0 and feat in sub.columns:
                ax.plot(sub["year"], sub[feat],
                        color=ALG_COLORS[alg], ls=ALG_STYLES[alg],
                        lw=ALG_WIDTH[alg],
                        label=alg if (ai==0 and ci==0) else None)
        ax.set_title(f"{ctry} – {feat.replace('_',' ')}", fontsize=7)
        ax.tick_params(labelsize=6); ax.grid(True, alpha=0.3)

fig.legend(loc="lower center", ncol=4, fontsize=8, frameon=True)
plt.tight_layout(rect=[0,0.03,1,1])
AM.save_figure(fig, "14_trajectories_2025_2050")
print("  → output/figures/14_trajectories_2025_2050.png")
print("✓ Trajectory simulation done")


Simulating trajectories for 26 years (2025-2050) ...
  ⏱    Traj Baseline: 0.3s
  ⏱    Traj MADDPG: 0.3s
  ⏱    Traj MATD3: 0.3s
  ⏱    Traj Evo-MATD3-SR: 0.3s
  Trajectory dataframe: (21632, 40)
  → output/figures/14_trajectories_2025_2050.png
✓ Trajectory simulation done


## Cell 23 · Russia Trajectory 2025-2050: Scenarios and Evolution Directions

In [23]:
# ── Russia 2025-2050 under different algorithms ───────────────────────────────
RUS_FEATS = [f for f in ["gdp_growth","governance_mean",
                          "trade_gdp","renewable_energy",
                          "internet_users","unemployment",
                          "gini_index","life_expectancy"]
             if f in FEAT_COLS][:6]

if RUS_CODE in countries:
    df_rus_traj = df_traj_all[df_traj_all["country_code"] == RUS_CODE].copy()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("Russia (RUS) — Trajectory 2025-2050 by Algorithm", fontsize=12)
    for ai, feat in enumerate(RUS_FEATS):
        ax = axes[ai // 3][ai % 3]
        for alg in ["Baseline","MADDPG","MATD3","Evo-MATD3-SR"]:
            sub = df_rus_traj[df_rus_traj["algorithm"]==alg]
            if len(sub) > 0 and feat in sub.columns:
                ax.plot(sub["year"], sub[feat],
                        color=ALG_COLORS[alg], ls=ALG_STYLES[alg],
                        lw=ALG_WIDTH[alg], label=alg if ai==0 else None)
        ax.set_title(feat.replace("_"," "), fontsize=8)
        ax.tick_params(labelsize=7); ax.grid(True, alpha=0.3)
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=8)
    plt.tight_layout(rect=[0,0.03,1,1])
    AM.save_figure(fig, "15_russia_trajectory")
    print("  → output/figures/15_russia_trajectory.png")

    # ── Russia under specific crisis scenarios (trajectory) ───────────────────
    RUS_CRISIS_TRAJ = []
    rus_ci = countries.index(RUS_CODE)

    def simulate_trajectory_crisis(actor, env, init_states, sim_years,
                                    feat_cols, country_list, crisis_obj,
                                    policy_label="Policy"):
        """Trajectory with a fixed crisis injected throughout."""
        records = []
        states  = env.reset(init_states.copy())
        env.crisis   = crisis_obj
        env.crisis_t = len(sim_years) + 10   # persistent crisis
        for yr in sim_years:
            st = torch.tensor(states, dtype=torch.float32, device=DEVICE)
            actor.eval()
            with torch.no_grad():
                acts = actor(st).cpu().numpy()
            next_s, rws, done, info = env.step(acts)
            for ri, ctry in enumerate(country_list):
                if ctry != RUS_CODE: continue
                row = {"country_code": ctry, "year": yr,
                       "algorithm": policy_label, "crisis": crisis_obj["name"],
                       "reward": float(rws[ri])}
                n_feats = min(len(feat_cols), states.shape[1])
                for fi, feat in enumerate(feat_cols[:n_feats]):
                    row[feat] = float(states[ri, fi])
                records.append(row)
            states = next_s
        return pd.DataFrame(records)

    # 4 Russia-relevant crises × Evo-MATD3-SR actor
    rus_crisis_selection = FULL_CRISIS_PALETTE[:4]   # first 4 for brevity
    for crisis in rus_crisis_selection:
        df_crt = simulate_trajectory_crisis(
            evo_actor, env_test, init_te, SIM_YEARS[:15],   # first 15 years
            FEAT_COLS, countries, crisis, policy_label="Evo-MATD3-SR")
        RUS_CRISIS_TRAJ.append(df_crt)

    df_rus_crisis_traj = pd.concat(RUS_CRISIS_TRAJ, ignore_index=True)
    AM.save_table(df_rus_crisis_traj, "16_russia_crisis_trajectories")

    if len(df_rus_crisis_traj) > 0 and "gdp_growth" in df_rus_crisis_traj.columns:
        fig2, ax2 = plt.subplots(figsize=(10, 5))
        for crisis in rus_crisis_selection:
            sub = df_rus_crisis_traj[df_rus_crisis_traj["crisis"]==crisis["name"]]
            if len(sub) > 0:
                ax2.plot(sub["year"], sub["gdp_growth"],
                         label=crisis["name"][:35], lw=1.8)
        ax2.set_title("Russia: GDP Growth under 4 Crisis Scenarios (Evo-MATD3-SR)")
        ax2.set_xlabel("Year"); ax2.set_ylabel("gdp_growth (normalised)")
        ax2.legend(fontsize=7); ax2.grid(True, alpha=0.4)
        plt.tight_layout()
        AM.save_figure(fig2, "16_russia_crisis_gdp_trajectories")
        print("  → output/figures/16_russia_crisis_gdp_trajectories.png")
else:
    print(f"  ⚠️  {RUS_CODE} not in dataset")
print("✓ Russia trajectory deep dive done")


  → output/figures/15_russia_trajectory.png
  → output/figures/16_russia_crisis_gdp_trajectories.png
✓ Russia trajectory deep dive done


## Cell 24 · Evolution Analysis: World Order Attractors and Direction Vectors

In [25]:
# ── Attractor analysis ────────────────────────────────────────────────────────
# Compare the end-state distribution of trajectories:
# Baseline vs MADDPG vs MATD3 vs Evo-MATD3-SR
# Cluster end-states to find "world order attractors"

FEATS_ATTRACTOR = [
    f for f in [
        "gdp_growth", "governance_mean", "trade_gdp",
        "gini_index", "renewable_energy", "internet_users"
    ]
    if f in FEAT_COLS
]


def compute_endstates(df_traj, year_end=2050, alg="Evo-MATD3-SR"):
    sub = df_traj[(df_traj["year"] == year_end) & (df_traj["algorithm"] == alg)].copy()
    avail = [f for f in FEATS_ATTRACTOR if f in sub.columns]
    if len(sub) == 0 or not avail:
        return pd.DataFrame()
    out = sub[["country_code"] + avail].drop_duplicates(subset=["country_code"]).set_index("country_code")
    out = out.replace([np.inf, -np.inf], np.nan).dropna(how="all")
    return out


endstates = {}
for alg in ["Baseline", "MADDPG", "MATD3", "Evo-MATD3-SR"]:
    es = compute_endstates(df_traj_all, year_end=SIM_YEARS[-1], alg=alg)
    if len(es) > 0:
        endstates[alg] = es

if endstates:
    from sklearn.preprocessing import StandardScaler

    alg_order = [a for a in ["Baseline", "MADDPG", "MATD3", "Evo-MATD3-SR"] if a in endstates]
    common_ctrs = sorted(list(set.intersection(*[set(es.index) for es in endstates.values()])))
    common_feats = [f for f in FEATS_ATTRACTOR if all(f in es.columns for es in endstates.values())]

    if common_ctrs and common_feats:
        # ── Build common matrix across all algorithms ────────────────────────
        mats = []
        for alg in alg_order:
            mat_alg = (
                endstates[alg]
                .loc[common_ctrs, common_feats]
                .copy()
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0.0)
            )
            mats.append(mat_alg.values)

        all_end = np.vstack(mats)

        scaler_end = StandardScaler()
        all_end_scaled = scaler_end.fit_transform(all_end)

        pca_end = PCA(n_components=2, random_state=SEED)
        pca_end.fit(all_end_scaled)

        fig, ax = plt.subplots(figsize=(10, 7))
        pal_alg = {
            "Baseline": "#aaaaaa",
            "MADDPG": "#4c72b0",
            "MATD3": "#dd8452",
            "Evo-MATD3-SR": "#55a868"
        }
        marks = {
            "Baseline": "x",
            "MADDPG": "o",
            "MATD3": "s",
            "Evo-MATD3-SR": "^"
        }

        coords_by_alg = {}

        for alg in alg_order:
            mat_alg = (
                endstates[alg]
                .loc[common_ctrs, common_feats]
                .copy()
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0.0)
                .values
            )
            mat_alg_scaled = scaler_end.transform(mat_alg)
            coords = pca_end.transform(mat_alg_scaled)
            coords_by_alg[alg] = coords

            ax.scatter(
                coords[:, 0], coords[:, 1],
                c=pal_alg[alg], marker=marks[alg], s=50,
                alpha=0.7, label=alg
            )

            # Centroid
            cx, cy = coords[:, 0].mean(), coords[:, 1].mean()
            ax.scatter(
                cx, cy,
                c=pal_alg[alg], marker="D", s=180,
                edgecolors="black", zorder=5
            )
            ax.annotate(
                f"{alg}\ncentroid",
                (cx, cy),
                xytext=(cx + 0.1, cy + 0.1),
                fontsize=7,
                color=pal_alg[alg]
            )

        # ── Arrow: direction of evolution (Baseline → Evo) ───────────────────
        if "Baseline" in coords_by_alg and "Evo-MATD3-SR" in coords_by_alg:
            base_c = coords_by_alg["Baseline"].mean(axis=0)
            evo_c = coords_by_alg["Evo-MATD3-SR"].mean(axis=0)

            ax.annotate(
                "",
                xy=evo_c,
                xytext=base_c,
                arrowprops=dict(arrowstyle="->", color="black", lw=2.5)
            )
            ax.text(
                (base_c[0] + evo_c[0]) / 2,
                (base_c[1] + evo_c[1]) / 2 - 0.3,
                "Evolution direction",
                fontsize=8,
                ha="center",
                style="italic"
            )

        ax.set_xlabel(f"PC1 ({pca_end.explained_variance_ratio_[0] * 100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca_end.explained_variance_ratio_[1] * 100:.1f}%)")
        ax.set_title(
            "World Order Attractors — End-State PCA (2050)\n"
            "Each point = one country; diamonds = algorithm centroids",
            fontsize=10
        )
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        AM.save_figure(fig, "17_world_order_attractors")
        print("  → output/figures/17_world_order_attractors.png")

        # ── Direction vectors: mean Δstate per algorithm ─────────────────────
        baseline_mean = (
            endstates["Baseline"]
            .loc[common_ctrs, common_feats]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
            .mean()
        )

        dir_df = pd.DataFrame({
            alg: (
                endstates[alg]
                .loc[common_ctrs, common_feats]
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0.0)
                .mean() - baseline_mean
            )
            for alg in alg_order if alg != "Baseline"
        }).round(4)

        AM.save_table(dir_df, "17_evolution_directions")
        AM.show_table(dir_df, "Evolution Direction Vectors (Δ from Baseline, 2050)")

        # ── K-means on end-states: world order clusters ──────────────────────
        evo_end_mat = endstates.get("Evo-MATD3-SR", endstates[alg_order[-1]])
        evo_end_mat = (
            evo_end_mat
            .loc[common_ctrs, common_feats]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

        if len(evo_end_mat) >= 4:
            km_wo = KMeans(n_clusters=4, n_init=20, random_state=SEED)
            wo_labels = km_wo.fit_predict(evo_end_mat)

            wo_df = pd.DataFrame({
                "country": evo_end_mat.index,
                "world_order_cluster": wo_labels
            })

            AM.save_table(wo_df, "18_world_order_clusters_2050")
            AM.show_table(wo_df, "World Order Clusters 2050 (K=4)")

            cluster_sizes = pd.Series(wo_labels).value_counts().sort_index()
            print("  World order cluster sizes:")
            for cl, sz in cluster_sizes.items():
                members = evo_end_mat.index[wo_labels == cl].tolist()[:6]
                print(f"    Cluster {cl}: {sz} countries — {members}")
    else:
        print("  ⚠️  No common countries/features across algorithms for attractor PCA")
else:
    print("  ⚠️  No end-states available for attractor analysis")

print("✓ World order attractor analysis done")

  → output/figures/17_world_order_attractors.png


,MADDPG,MATD3,Evo-MATD3-SR
gdp_growth,0.0304,0.0227,0.0657
trade_gdp,-0.0261,-0.0321,0.0047
gini_index,0.0152,0.0160,0.0213
renewable_energy,0.0092,0.0098,0.0055
internet_users,0.0096,0.0092,0.0210


,country,world_order_cluster
0,ABW,0
1,AFG,1
2,AGO,1
3,ALB,0
4,AND,0
5,ARE,3
6,ARG,1
7,ARM,0
8,ATG,0
9,AUS,0


  World order cluster sizes:
    Cluster 0: 135 countries — ['ABW', 'ALB', 'AND', 'ARM', 'ATG', 'AUS']
    Cluster 1: 64 countries — ['AFG', 'AGO', 'ARG', 'BDI', 'BEN', 'BFA']
    Cluster 2: 1 countries — ['LIE']
    Cluster 3: 8 countries — ['ARE', 'DJI', 'HKG', 'IRL', 'LUX', 'MLT']
✓ World order attractor analysis done


## Cell 25 · Training Diagnostics: Learning Curves, Reward & Loss

In [26]:
# ── Load logs ─────────────────────────────────────────────────────────────────
maddpg_df = maddpg_log.to_df()
matd3_df  = matd3_log.to_df()
evo_df    = evo_bias_log  # already a DataFrame

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Training Diagnostics — All Algorithms", fontsize=12, fontweight="bold")

# MADDPG reward
ax = axes[0][0]
if "reward" in maddpg_df.columns:
    smooth = maddpg_df["reward"].rolling(20, min_periods=1).mean()
    ax.fill_between(maddpg_df["episode"],
                    maddpg_df["reward"].rolling(20,min_periods=1).min(),
                    maddpg_df["reward"].rolling(20,min_periods=1).max(),
                    alpha=0.15, color="#4c72b0")
    ax.plot(maddpg_df["episode"], maddpg_df["reward"], alpha=0.3, color="#4c72b0", lw=0.7)
    ax.plot(maddpg_df["episode"], smooth, color="#4c72b0", lw=2, label="MADDPG")
ax.set_title("MADDPG – Reward"); ax.set_xlabel("Episode"); ax.legend(); ax.grid(True,alpha=0.4)

# MATD3 reward
ax = axes[0][1]
if "reward" in matd3_df.columns:
    smooth = matd3_df["reward"].rolling(20, min_periods=1).mean()
    ax.fill_between(matd3_df["episode"],
                    matd3_df["reward"].rolling(20,min_periods=1).min(),
                    matd3_df["reward"].rolling(20,min_periods=1).max(),
                    alpha=0.15, color="#dd8452")
    ax.plot(matd3_df["episode"], matd3_df["reward"], alpha=0.3, color="#dd8452", lw=0.7)
    ax.plot(matd3_df["episode"], smooth, color="#dd8452", lw=2, label="MATD3")
ax.set_title("MATD3 – Reward"); ax.set_xlabel("Episode"); ax.legend(); ax.grid(True,alpha=0.4)

# Evo convergence
ax = axes[0][2]
if "score_mean" in evo_df.columns:
    ax.fill_between(evo_df["iter"],
                    evo_df["score_mean"] - evo_df["score_std"],
                    evo_df["score_mean"] + evo_df["score_std"],
                    alpha=0.2, color="#55a868")
    ax.plot(evo_df["iter"], evo_df["score_mean"], color="#55a868", lw=2, label="Evo mean")
    ax.plot(evo_df["iter"], evo_df["score_best"], color="darkgreen", ls="--", lw=1.5,
            label="Evo best")
ax.set_title("Evo-MATD3-SR – Reward"); ax.set_xlabel("Iteration")
ax.legend(); ax.grid(True, alpha=0.4)

# MADDPG critic loss
ax = axes[1][0]
if "loss_c" in maddpg_df.columns:
    ax.plot(maddpg_df["episode"], maddpg_df["loss_c"].rolling(20,min_periods=1).mean(),
            color="#4c72b0", lw=1.5)
ax.set_title("MADDPG – Critic Loss"); ax.set_xlabel("Episode"); ax.grid(True,alpha=0.4)

# MATD3 critic & actor loss
ax = axes[1][1]
if "loss_c" in matd3_df.columns:
    ax.plot(matd3_df["episode"], matd3_df["loss_c"].rolling(20,min_periods=1).mean(),
            color="#dd8452", lw=1.5, label="Critic")
if "loss_a" in matd3_df.columns:
    ax.plot(matd3_df["episode"], matd3_df["loss_a"].rolling(20,min_periods=1).mean(),
            color="coral", lw=1.5, ls="--", label="Actor")
ax.set_title("MATD3 – Critic/Actor Loss"); ax.set_xlabel("Episode")
ax.legend(); ax.grid(True, alpha=0.4)

# Evo sigma decay
ax = axes[1][2]
if "sigma_mean" in evo_df.columns:
    ax.plot(evo_df["iter"], evo_df["sigma_mean"], color="#55a868", lw=1.5)
ax.set_title("Evo-MATD3-SR – σ Decay (exploration)"); ax.set_xlabel("Iteration")
ax.grid(True, alpha=0.4)

plt.tight_layout()
AM.save_figure(fig, "18_training_diagnostics")
print("  → output/figures/18_training_diagnostics.png")
print("✓ Training diagnostics done")


  → output/figures/18_training_diagnostics.png
✓ Training diagnostics done


## Cell 26 · Country Scenario Analysis: 5 Macro Scenarios 2025-2050

In [27]:
# ── 5 future macro-scenarios ──────────────────────────────────────────────────
MACRO_SCENARIOS = {
    "S1_Cooperation":     # Deep multilateral cooperation
        {"gdp_mult":1.05, "gov_mult":1.08, "energy_mult":1.10,
         "description": "Multilateral cooperation, strong institutions, green transition"},
    "S2_Fragmentation":   # Geopolitical bloc fragmentation
        {"gdp_mult":0.92, "gov_mult":0.88, "energy_mult":0.85,
         "description": "Bloc-based fragmentation, sanctions, limited trade"},
    "S3_ClimatePath":     # Climate-driven transformation
        {"gdp_mult":0.97, "gov_mult":1.02, "energy_mult":1.30,
         "description": "Aggressive climate action, energy transition, moderate growth"},
    "S4_TechLeap":        # Technology & digitalisation surge
        {"gdp_mult":1.08, "gov_mult":1.05, "energy_mult":1.15,
         "description": "AI/digital surge, labour disruption, innovation-led growth"},
    "S5_CascadeCrisis":   # Compounding crises
        {"gdp_mult":0.85, "gov_mult":0.80, "energy_mult":0.75,
         "description": "Compounding financial/climate/political crises"},
}

def simulate_macro_scenario(actor, env, init_states, sim_years,
                              feat_cols, country_list, scenario_name,
                              gdp_mult=1.0, gov_mult=1.0, energy_mult=1.0):
    """Apply scenario multipliers to reward weights; simulate trajectory."""
    # Temporarily modify env reward weights
    orig_weights = copy.deepcopy(env.cfg.REWARD_WEIGHTS)
    env.cfg.REWARD_WEIGHTS["gdp_growth"]       *= gdp_mult
    env.cfg.REWARD_WEIGHTS["governance_mean"]  *= gov_mult
    env.cfg.REWARD_WEIGHTS["renewable_energy"] *= energy_mult
    # Renormalise
    total = sum(env.cfg.REWARD_WEIGHTS.values())
    for k in env.cfg.REWARD_WEIGHTS:
        env.cfg.REWARD_WEIGHTS[k] /= total

    records = []
    states  = env.reset(init_states.copy())
    for yr in sim_years:
        st = torch.tensor(states, dtype=torch.float32, device=DEVICE)
        actor.eval()
        with torch.no_grad():
            acts = actor(st).cpu().numpy()
        next_s, rws, done, info = env.step(acts)
        for ri, ctry in enumerate(country_list):
            row = {"country_code": ctry, "year": yr,
                   "scenario": scenario_name, "reward": float(rws[ri])}
            if feat_cols and "gdp_growth" in feat_cols:
                fi_g = feat_cols.index("gdp_growth") if "gdp_growth" in feat_cols else -1
                row["gdp_growth"] = float(states[ri, fi_g]) if fi_g >= 0 else np.nan
            records.append(row)
        states = next_s

    env.cfg.REWARD_WEIGHTS = orig_weights   # Restore
    return pd.DataFrame(records)

print("Simulating 5 macro-scenarios (Evo-MATD3-SR) 2025-2040...")
scenario_dfs = []
for sc_name, sc_params in MACRO_SCENARIOS.items():
    df_sc = simulate_macro_scenario(
        evo_actor, env_test, init_te, SIM_YEARS[:16],   # 2025-2040
        FEAT_COLS, countries, sc_name,
        gdp_mult    = sc_params["gdp_mult"],
        gov_mult    = sc_params["gov_mult"],
        energy_mult = sc_params["energy_mult"],
    )
    scenario_dfs.append(df_sc)
    print(f"  {sc_name}: done")

df_scenarios = pd.concat(scenario_dfs, ignore_index=True)
AM.save_table(df_scenarios, "19_macro_scenarios_2025_2040")

# ── Aggregate reward by scenario × year ──────────────────────────────────────
sc_agg = df_scenarios.groupby(["scenario","year"])["reward"].mean().reset_index()
fig, ax = plt.subplots(figsize=(12, 6))
SC_COLORS = plt.cm.tab10(np.linspace(0,1,5))
for i, sc in enumerate(MACRO_SCENARIOS.keys()):
    sub = sc_agg[sc_agg["scenario"]==sc]
    desc = MACRO_SCENARIOS[sc]["description"]
    ax.plot(sub["year"], sub["reward"], color=SC_COLORS[i], lw=2,
            label=f"{sc}: {desc[:35]}")
ax.axhline(0, color="gray", ls="--", lw=1)
ax.set_xlabel("Year"); ax.set_ylabel("Mean Global Reward")
ax.set_title("5 Macro-Scenarios: Global Mean Reward 2025-2040 (Evo-MATD3-SR)")
ax.legend(fontsize=7, loc="lower right")
ax.grid(True, alpha=0.4)
plt.tight_layout()
AM.save_figure(fig, "19_macro_scenarios_reward")
print("  → output/figures/19_macro_scenarios_reward.png")

# ── Russia under 5 scenarios ──────────────────────────────────────────────────
if RUS_CODE in countries:
    df_rus_sc = df_scenarios[df_scenarios["country_code"]==RUS_CODE]
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    for i, sc in enumerate(MACRO_SCENARIOS.keys()):
        sub = df_rus_sc[df_rus_sc["scenario"]==sc]
        ax2.plot(sub["year"], sub["reward"], color=SC_COLORS[i], lw=2, label=sc)
    ax2.set_title("Russia (RUS) — Reward under 5 Macro-Scenarios (Evo-MATD3-SR)")
    ax2.set_xlabel("Year"); ax2.set_ylabel("Reward")
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.4)
    plt.tight_layout()
    AM.save_figure(fig2, "20_russia_macro_scenarios")
    print("  → output/figures/20_russia_macro_scenarios.png")

AM.show_table(
    sc_agg.pivot(index="year", columns="scenario", values="reward").round(4),
    "Macro-Scenarios: Mean Global Reward by Year")
print("✓ Country scenario analysis done")


Simulating 5 macro-scenarios (Evo-MATD3-SR) 2025-2040...
  S1_Cooperation: done
  S2_Fragmentation: done
  S3_ClimatePath: done
  S4_TechLeap: done
  S5_CascadeCrisis: done
  → output/figures/19_macro_scenarios_reward.png
  → output/figures/20_russia_macro_scenarios.png


scenario,S1_Cooperation,S2_Fragmentation,S3_ClimatePath,S4_TechLeap,S5_CascadeCrisis
year,,,,,
2025,0.0186,0.0178,0.0177,0.0192,0.0194
2026,0.0177,0.0178,0.0179,0.0188,0.0180
2027,0.0177,0.0181,0.0181,0.0190,0.0182
2028,0.0182,0.0179,0.0191,0.0190,0.0192
2029,0.0186,0.0183,0.0189,0.0186,0.0188
2030,0.0186,0.0177,0.0188,0.0187,0.0177
2031,0.0183,0.0191,0.0187,0.0182,0.0175
2032,0.0182,0.0188,0.0188,0.0180,0.0175
2033,0.0178,0.0184,0.0180,0.0180,0.0175


✓ Country scenario analysis done


## Cell 27 · Final Dashboard: Comprehensive Summary

In [29]:
# ── Comprehensive 4×4 dashboard ──────────────────────────────────────────────
plt.style.use(CFG.FIGURE_STYLE)
fig = plt.figure(figsize=(20, 18))
gs  = gridspec.GridSpec(4, 4, figure=fig, hspace=0.45, wspace=0.4)
fig.suptitle(
    "World Political System as MARL — Comprehensive Dashboard\n"
    "WDI+WGI Panel 1990-2023 | Forecast Horizon 2025-2050 | "
    "MADDPG / MATD3 / Evo-MATD3-SR",
    fontsize=12, fontweight="bold"
)

# ── Panel 0,0 — Algorithm comparison bar ─────────────────────────────────────
ax00 = fig.add_subplot(gs[0, 0])
alg_bar_data = [(r["label"], r["mean_rew"], r["std_rew"]) for r in results]
labs, ms, ss = zip(*alg_bar_data)
cols = ["#aaaaaa", "#4c72b0", "#dd8452", "#55a868"]
ax00.bar(labs, ms, yerr=ss, capsize=4, color=cols, alpha=0.85, edgecolor="black")
ax00.set_title("Algorithm Comparison\n(test reward)", fontsize=8)
ax00.tick_params(axis="x", rotation=20, labelsize=6)
ax00.set_ylabel("Reward", fontsize=7)
ax00.grid(True, alpha=0.3, axis="y")

# ── Panel 0,1 — Evo bias vector ──────────────────────────────────────────────
ax01 = fig.add_subplot(gs[0, 1])
bias_vals = evo_actor.bias.data.cpu().numpy()
cols_bias = ["#dd4444" if v < 0 else "#44aa44" for v in bias_vals]
ax01.bar(range(CFG.ACTION_DIM), bias_vals, color=cols_bias, alpha=0.8)
ax01.set_xticks(range(CFG.ACTION_DIM))
ax01.set_xticklabels([f"A{i+1}" for i in range(CFG.ACTION_DIM)],
                     rotation=60, fontsize=6)
ax01.axhline(0, color="black", lw=0.8)
ax01.set_title("Evo Action Bias\n(Evo-MATD3-SR)", fontsize=8)
ax01.set_ylabel("Bias value", fontsize=7)
ax01.grid(True, alpha=0.3, axis="y")

# ── Panel 0,2-3 — World Model MAE ────────────────────────────────────────────
ax02 = fig.add_subplot(gs[0, 2:4])
top_mae = wm_mae_df.sort_values("MAE").head(10)
ax02.barh(top_mae["feature"], top_mae["MAE"],
          color="steelblue", alpha=0.8)
ax02.set_title("World Model — MAE (best 10 features, val set)", fontsize=8)
ax02.tick_params(labelsize=7)
ax02.grid(True, alpha=0.3, axis="x")

# ── Panel 1,0-1 — MADDPG/MATD3 reward curves ─────────────────────────────────
ax10 = fig.add_subplot(gs[1, 0:2])
if "reward" in maddpg_df.columns:
    ax10.plot(
        maddpg_df["episode"],
        maddpg_df["reward"].rolling(25, min_periods=1).mean(),
        color="#4c72b0", lw=1.5, label="MADDPG"
    )
if "reward" in matd3_df.columns:
    ax10.plot(
        matd3_df["episode"],
        matd3_df["reward"].rolling(25, min_periods=1).mean(),
        color="#dd8452", lw=1.5, label="MATD3"
    )
ax10.set_title("Training Reward Curves (MA-25)", fontsize=8)
ax10.set_xlabel("Episode", fontsize=7)
ax10.set_ylabel("Reward", fontsize=7)
ax10.legend(fontsize=7)
ax10.grid(True, alpha=0.3)

# ── Panel 1,2-3 — Crisis heatmap (subset) ────────────────────────────────────
ax12 = fig.add_subplot(gs[1, 2:4])
stress_sub = df_stress_all[df_stress_all["algorithm"] == "Evo-MATD3-SR"]
stress_sub10 = (
    stress_sub.sort_values("mean_rew")
    .head(10)[["crisis_name", "mean_rew"]]
)
ax12.barh(stress_sub10["crisis_name"], stress_sub10["mean_rew"],
          color="tomato", alpha=0.85)
ax12.set_title("10 Most Damaging Crises\n(Evo-MATD3-SR reward)", fontsize=8)
ax12.tick_params(labelsize=6)
ax12.grid(True, alpha=0.3, axis="x")
ax12.axvline(0, color="black", lw=0.8)

# ── Panel 2,0-1 — Russia trajectory ─────────────────────────────────────────
ax20 = fig.add_subplot(gs[2, 0:2])
if RUS_CODE in countries and "gdp_growth" in FEAT_COLS:
    for alg in ["Baseline", "MADDPG", "MATD3", "Evo-MATD3-SR"]:
        sub = df_traj_all[
            (df_traj_all["country_code"] == RUS_CODE) &
            (df_traj_all["algorithm"] == alg)
        ]
        if len(sub) > 0:
            feat_plot = "gdp_growth" if "gdp_growth" in sub.columns else sub.columns[4]
            ax20.plot(
                sub["year"], sub[feat_plot],
                color=ALG_COLORS[alg], ls=ALG_STYLES[alg],
                lw=ALG_WIDTH[alg], label=alg
            )
ax20.set_title("Russia (RUS) — GDP Growth 2025-2050", fontsize=8)
ax20.set_xlabel("Year", fontsize=7)
ax20.legend(fontsize=6)
ax20.grid(True, alpha=0.3)

# ── Panel 2,2-3 — Macro-scenarios ───────────────────────────────────────────
ax22 = fig.add_subplot(gs[2, 2:4])
SC_COLORS2 = plt.cm.Set1(np.linspace(0, 0.9, 5))
for i, sc in enumerate(MACRO_SCENARIOS.keys()):
    sub = sc_agg[sc_agg["scenario"] == sc]
    ax22.plot(
        sub["year"], sub["reward"],
        color=SC_COLORS2[i], lw=1.8,
        label=sc.replace("S", "").replace("_", " ")
    )
ax22.axhline(0, color="gray", ls="--", lw=0.8)
ax22.set_title("5 Macro-Scenarios — Global Reward", fontsize=8)
ax22.set_xlabel("Year", fontsize=7)
ax22.legend(fontsize=6, ncol=2)
ax22.grid(True, alpha=0.3)

# ── Panel 3,0-1 — G7 vs BRICS rewards ───────────────────────────────────────
ax30 = fig.add_subplot(gs[3, 0:2])
if len(rew_df) > 0:
    rew_df_sorted = rew_df.sort_values("mean_reward", ascending=True)
    cols_gb = ["#4c72b0" if g == "G7" else "#dd8452"
               for g in rew_df_sorted["group"]]
    ax30.barh(
        rew_df_sorted["country"], rew_df_sorted["mean_reward"],
        color=cols_gb, alpha=0.85
    )
ax30.set_title("G7 (blue) vs BRICS+ (orange)\nMean Reward (Evo-MATD3-SR)", fontsize=8)
ax30.tick_params(labelsize=7)
ax30.grid(True, alpha=0.3, axis="x")
ax30.axvline(0, color="black", lw=0.8)

# ── Panel 3,2-3 — Vulnerability vs reward scatter ───────────────────────────
ax32 = fig.add_subplot(gs[3, 2:4])
vuln_vals  = [VULN_MAP.get(c, 0.5) for c in spotlight]
rew_vals_s = [ctry_rews_evo.get(c, 0) for c in spotlight]
colors_sp  = [pal7[CLUSTER_MAP.get(c, 0)] for c in spotlight]
ax32.scatter(
    vuln_vals, rew_vals_s,
    c=colors_sp, s=80, alpha=0.8, edgecolors="black"
)
for i, c in enumerate(spotlight):
    ax32.annotate(c, (vuln_vals[i], rew_vals_s[i]), fontsize=7, alpha=0.8)
ax32.set_xlabel("Vulnerability Index", fontsize=7)
ax32.set_ylabel("Mean Reward (Evo)", fontsize=7)
ax32.set_title("Vulnerability vs Reward\n(spotlight countries)", fontsize=8)
ax32.grid(True, alpha=0.3)

AM.save_figure(fig, "21_final_dashboard")
print("  → output/figures/21_final_dashboard.png")
print("✓ Final dashboard done")

  → output/figures/21_final_dashboard.png
✓ Final dashboard done


## Cell 28 · Interactive Tables (ipywidgets, Colab + GitHub HTML export)

In [30]:
# ── 28.1 Interactive widget table builder ────────────────────────────────────
def make_widget_table(df: pd.DataFrame, title: str = "",
                      max_rows: int = 200) -> widgets.VBox:
    """Create searchable/filterable HTML widget table for Colab.
    Compatible with nbconvert HTML export (visible on GitHub Pages).
    """
    df_sub = df.head(max_rows).copy()

    # Convert to HTML
    html_str = df_sub.to_html(border=1, classes="widget-table",
                               float_format=lambda x: f"{x:.4f}",
                               index=True)
    html_full = f"""
    <style>
      .widget-table {{ border-collapse:collapse; font-size:11px;
                       font-family:monospace; width:100%; }}
      .widget-table th {{ background:#2c3e50; color:white;
                          padding:4px 8px; text-align:left; }}
      .widget-table td {{ padding:3px 8px; border-bottom:1px solid #ddd; }}
      .widget-table tr:nth-child(even) {{ background:#f5f5f5; }}
      .widget-table tr:hover {{ background:#e8f4fd; }}
      #search_{id(df)} {{ margin-bottom:8px; padding:4px;
                          width:300px; font-size:12px; }}
    </style>
    <h4 style='font-family:sans-serif'>{title} ({len(df)} rows total, showing {len(df_sub)})</h4>
    <input id='search_{id(df)}' type='text' placeholder='🔍 Filter...'
           oninput='filterTable_{id(df)}(this.value)' />
    <div style='overflow:auto; max-height:400px'>
      <div id='table_container_{id(df)}'>{html_str}</div>
    </div>
    <script>
    function filterTable_{id(df)}(query) {{
      query = query.toLowerCase();
      var rows = document.getElementById('table_container_{id(df)}')
                         .querySelectorAll('tr');
      rows.forEach(function(row, idx) {{
        if (idx === 0) return;
        var text = row.innerText.toLowerCase();
        row.style.display = text.includes(query) ? '' : 'none';
      }});
    }}
    </script>
    """
    out = widgets.Output()
    with out:
        display(HTML(html_full))
    return widgets.VBox([out])

# ── 28.2 Load CSV outputs and display as widgets ─────────────────────────────
import glob
tables_dir = Path("output/tables")
csv_files  = sorted(tables_dir.glob("*.csv"))
print(f"  Found {len(csv_files)} output tables")

# Key tables to display
KEY_TABLES = {
    "06_algorithm_comparison":  "Algorithm Comparison",
    "08_crisis_stress_test":    "Crisis Stress Test (30 scenarios)",
    "09_country_action_profiles": "Country Action Profiles",
    "14_hypothesis_list":       "Political Hypotheses",
    "17_evolution_directions":  "Evolution Direction Vectors",
    "18_world_order_clusters_2050": "World Order Clusters 2050",
}

for fname, title in KEY_TABLES.items():
    fpath = tables_dir / f"{fname}.csv"
    if fpath.exists():
        df_t = pd.read_csv(fpath, index_col=0)
        widget = make_widget_table(df_t, title, max_rows=100)
        display(widget)
    else:
        print(f"  ⚠️  Table not found: {fpath}")

# ── 28.3 Export summary manifest ─────────────────────────────────────────────
figures_dir  = Path("output/figures")
checkpts_dir = Path("output/checkpoints")

manifest = {
    "pipeline": "World Political System as MARL (WDI+WGI)",
    "n_countries":  N_COUNTRIES,
    "state_dim":    CFG.STATE_DIM,
    "action_dim":   CFG.ACTION_DIM,
    "forecast_horizon": "2025-2050",
    "crisis_palette": len(FULL_CRISIS_PALETTE),
    "hypotheses":   len(HYPOTHESES),
    "macro_scenarios": list(MACRO_SCENARIOS.keys()),
    "tables":  [f.name for f in sorted(tables_dir.glob("*.csv"))],
    "figures": [f.name for f in sorted(figures_dir.glob("*.png"))],
    "checkpoints": [f.name for f in sorted(checkpts_dir.glob("*.pt"))],
    "algorithm_results": {r["label"]: {"mean_rew": r["mean_rew"],
                                       "std_rew":  r["std_rew"]}
                          for r in results},
}
with open("output/pipeline_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("\n" + "═"*65)
print("  PIPELINE COMPLETE")
print("═"*65)
print(f"  Countries: {N_COUNTRIES}")
print(f"  State dim: {CFG.STATE_DIM}  Action dim: {CFG.ACTION_DIM}")
print(f"  Crisis palette: {len(FULL_CRISIS_PALETTE)} scenarios")
print(f"  Forecast horizon: 2025-2050")
print(f"  Hypotheses tested: {len(HYPOTHESES)}")
print(f"  Macro-scenarios: {len(MACRO_SCENARIOS)}")
print()
print("  Test-set results:")
for r in results:
    print(f"    {r['label']:30s}  {r['mean_rew']:+.4f} ± {r['std_rew']:.4f}")
print()
print("  Output tables:     output/tables/*.csv")
print("  Output figures:    output/figures/*.png")
print("  Checkpoints:       output/checkpoints/*.pt")
print("  Logs:              output/logs/*.json")
print("  Manifest:          output/pipeline_manifest.json")
print("═"*65)


  Found 20 output tables



═════════════════════════════════════════════════════════════════
  PIPELINE COMPLETE
═════════════════════════════════════════════════════════════════
  Countries: 208
  State dim: 35  Action dim: 14
  Crisis palette: 30 scenarios
  Forecast horizon: 2025-2050
  Hypotheses tested: 12
  Macro-scenarios: 5

  Test-set results:
    Baseline (no action)            +0.2644 ± 0.0016
    MADDPG                          +0.2766 ± 0.0020
    MATD3                           +0.2670 ± 0.0012
    Evo-MATD3-SR                    +0.3136 ± 0.0010

  Output tables:     output/tables/*.csv
  Output figures:    output/figures/*.png
  Checkpoints:       output/checkpoints/*.pt
  Logs:              output/logs/*.json
  Manifest:          output/pipeline_manifest.json
═════════════════════════════════════════════════════════════════


## Cell 29 · GitHub Export Helper

In [31]:
# ── Copy all outputs to a GitHub-ready directory ─────────────────────────────
import shutil

GITHUB_DIR = Path("world_politics_marl_export")
GITHUB_DIR.mkdir(exist_ok=True)
(GITHUB_DIR / "tables").mkdir(exist_ok=True)
(GITHUB_DIR / "figures").mkdir(exist_ok=True)

# Copy tables
for f in Path("output/tables").glob("*.csv"):
    shutil.copy(f, GITHUB_DIR / "tables" / f.name)

# Copy figures
for f in Path("output/figures").glob("*.png"):
    shutil.copy(f, GITHUB_DIR / "figures" / f.name)

# Copy manifest
shutil.copy("output/pipeline_manifest.json", GITHUB_DIR / "pipeline_manifest.json")

# ── Generate README.md ────────────────────────────────────────────────────────
readme = """
# World Political System as MARL
## WDI+WGI Panel | MADDPG · MATD3 · Evo-MATD3-SR | Forecast 2025-2050

### Pipeline summary
| Parameter | Value |
|-----------|-------|
| Countries | ~ 150+ (WDI/WGI coverage ≥ 30%) |
| State dimension | 41 (35 WDI + 6 WGI) |
| Action instruments | 14 policy instruments |
| Crisis scenarios | 30 (15 past + 15 future) |
| Forecast horizon | 2025 – 2050 |
| Hypotheses | 12 (incl. 2 Russia-specific) |
| Macro-scenarios | 5 |

### Algorithms
- **MADDPG** — cooperative multi-agent with shared actor/critic
- **MATD3** — twin-critic, policy delay, target noise
- **Evo-MATD3-SR** — evolutionary action-bias self-referential fine-tuning

### File structure
```
tables/     — CSV output tables (all widget tables)
figures/    — PNG figures (all charts/dashboards)
pipeline_manifest.json — full run manifest
```

### Reproduce
1. Upload `wdi_panel_raw.csv` and `wgidataset_with_sourcedata-2025.xlsx`
2. Open `World_Politics_MARL.ipynb` in Google Colab
3. Runtime → Run all

> All governance indicators sourced from World Bank WGI 2025.
> All WDI data sourced from World Bank Open Data.
> Crisis scenarios calibrated from IMF/WB historical data and IPCC AR6.
"""
with open(GITHUB_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme.strip())

# ── ZIP for download ──────────────────────────────────────────────────────────
zip_path = "world_politics_marl_export"
shutil.make_archive(zip_path, "zip", GITHUB_DIR)
print(f"  Created: {zip_path}.zip")

# Download (Colab)
try:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("  Download started (Colab)")
except Exception:
    print("  (Not in Colab – find the .zip in the working directory)")

print("\n✓ Export done — ready for GitHub upload")


  Created: world_politics_marl_export.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Download started (Colab)

✓ Export done — ready for GitHub upload


## References & Scientific Notes

### Data Sources
- **World Development Indicators (WDI)**: World Bank Open Data, 2024 edition.  
  https://databank.worldbank.org/source/world-development-indicators
- **Worldwide Governance Indicators (WGI)**: Kaufmann D., Kraay A., Mastruzzi M.  
  *Governance Matters — WGI 2025*, World Bank.  
  https://www.govindicators.org

### Methodological References
1. Lowe R. *MADDPG*: Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments. NeurIPS 2017.
2. Fujimoto S., van Hoof H., Meger D. *TD3*: Addressing Function Approximation Error in Actor-Critic Methods. ICML 2018.
3. Ha D., Schmidhuber J. *World Models* (Recurrent World Models Facilitate Policy Evolution). NeurIPS 2018.
4. Salimans T. et al. *Evolution Strategies as a Scalable Alternative to Reinforcement Learning*. OpenAI 2017.
5. Kaufmann D., Kraay A. *Worldwide Governance Indicators: Methodology and Analytical Issues*. Hague Journal on Rule of Law, 2011.
6. World Bank. *World Development Report 2021: Data for Better Lives*.
7. IPCC AR6 Working Group III, 2022. *Mitigation of Climate Change*.
8. WEF *Global Risks Report 2025*.

### Notes on Political Correctness & Scientific Framing
All country-level analysis in this work is conducted within a strictly
empirical framework. Governance scores (WGI) are statistical estimates from
the World Bank and represent institutional performance as measured by
composite survey data; they do not constitute normative political judgements
by the authors.

Crisis scenario labels (e.g. 'Sanctions Regime', 'Energy Shock') refer
to abstract macroeconomic shock typologies derived from empirical literature
and are applied uniformly to all countries in the model.

Russia (RUS) is analysed as one of 150+ countries in the dataset using the
same methodology, and no scenario is uniquely attributed to any single state.
Country-specific hypotheses (H11_RUS, H12_RUS) are scientifically motivated
by Russia's structural economic characteristics (energy export dependence)
and are analytically comparable to hypotheses one could formulate for,
e.g., Saudi Arabia or Norway.

The author do not express political views through this model.
